In [2]:
#Features engineering

In [1]:
import pandas as pd
import numpy as np

features = pd.read_csv('data/processed/features.csv')
features['tower_density_ratio'] = features['tower_count_1km'] / (features['tower_count_5km'] + 1)

# 2. Speed efficiency (upload/download ratio)
features['speed_ratio'] = features['avg_u_mbps'] / (features['avg_d_mbps'] + 1)

# 3. Rolling area average speed (temporal context)
features = features.sort_values(['area', 'year', 'quarter'])
features['area_rolling_d_mbps'] = (
    features.groupby('area')['avg_d_mbps']
    .transform(lambda x: x.shift(1).rolling(2, min_periods=1).mean())
    .fillna(features['avg_d_mbps'].mean())
)

# 4. Year trend (linear time signal)
features['year_trend'] = features['year'] - features['year'].min()

features.to_csv('data/processed/features_engineered.csv', index=False)
print("Engineered features shape:", features.shape)

Engineered features shape: (6086, 27)


In [27]:
# In your feature engineering notebook, add before saving:

# 1. Log-transform download speed target (right-skewed distribution)
# Don't add as feature — apply during training. But add these:

# 2. Area-level historical performance (very powerful signal)
area_stats = features.groupby('area').agg(
    area_median_d   = ('avg_d_mbps', 'median'),
    area_median_u   = ('avg_u_mbps', 'median'),
    area_median_lat = ('avg_lat_ms', 'median'),
    area_test_count = ('tests', 'sum')
).reset_index()
features = features.merge(area_stats, on='area', how='left')

# 3. Tower proximity interaction
features['distance_x_density'] = (
    features['nearest_tower_distance_km'] * 
    (1 / (features['tower_count_5km'] + 1))
)

features.to_csv('data/processed/features_engineered.csv', index=False)

In [3]:
#Training

In [28]:
import pandas as pd
import numpy as np
import json, os

features = pd.read_csv('data/processed/features_engineered.csv')

FEATURE_COLS = [
    'latitude', 'longitude',
    'nearest_tower_distance_km',
    'tower_count_1km', 'tower_count_2km', 'tower_count_5km',
    'tower_density_ratio',
    'digital_elevation_model',
    'region_enc', 'typeOfArea_enc', 'city_enc',
    'demand_growth_pct',
    'year_trend', 'quarter',
    'area_rolling_d_mbps',
    'tests',
    'area_median_d',
    'area_median_u', 
    'area_median_lat',
    'area_test_count',
    'distance_x_density',
]

TARGETS = ['avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms']

X = features[FEATURE_COLS]
y = features[TARGETS]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing in X:", X.isnull().sum().sum())

X shape: (6086, 21)
y shape: (6086, 3)
Missing in X: 0


In [7]:
from sklearn.model_selection import train_test_split

train_split = (features["year"] < 2024) | ((features["year"] == 2024) & (features["quarter"] <= 2))
val_split = (features["year"] == 2024) & (features["quarter"] >= 3)
test_split = (features["year"] == 2025)

X_train = X.loc[train_split]
X_val   = X.loc[val_split]
X_test  = X.loc[test_split] 
Y_train = y.loc[train_split]
Y_val   = y.loc[val_split]
Y_test  = y.loc[test_split]

print(f"Train: {X_train.shape[0]} rows | val: {X_val.shape[0]} | Test: {X_test.shape[0]} rows")

Train: 4577 rows | val: 506 | Test: 1003 rows


In [8]:
from sklearn.preprocessing import StandardScaler
import pickle

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled  = scaler.transform(X_val)

os.makedirs('outputs/models', exist_ok=True)
with open('outputs/models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler saved.")

Scaler saved.


In [9]:
#training

In [13]:
# scripts/train.py
import numpy as np
import pandas as pd
import pickle, json, os
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import sys
sys.path.insert(0, os.path.dirname('scripts/architecture.py'))
from architecture import NeuralNetworkRegressor

# ── Config ──────────────────────────────────────────────────────────────────
EPOCHS      = 200
BATCH_SIZE  = 64
LR          = 0.001
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

FEATURE_COLS = [
    'latitude', 'longitude',
    'nearest_tower_distance_km',
    'tower_count_1km', 'tower_count_2km', 'tower_count_5km',
    'tower_density_ratio',
    'digital_elevation_model',
    'region_enc', 'typeOfArea_enc', 'city_enc',
    'demand_growth_pct',
    'year_trend', 'quarter',
    'area_rolling_d_mbps',
    'tests'
]
TARGETS = ['avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms']

# ── Load data ────────────────────────────────────────────────────────────────
features = pd.read_csv('data/processed/features_engineered.csv')
X = features[FEATURE_COLS].values.astype(float)
y = features[TARGETS].values.astype(float)

# ── Temporal train/test split ────────────────────────────────────────────────
split_idx = int(len(X) * 0.8)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

# ── Scale ────────────────────────────────────────────────────────────────────
scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_val_s  = scaler_X.transform(X_val)

scaler_y = StandardScaler()
y_train_s = scaler_y.fit_transform(y_train)

os.makedirs('outputs/models', exist_ok=True)
with open('outputs/models/scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)
with open('outputs/models/scaler_y.pkl', 'wb') as f:
    pickle.dump(scaler_y, f)

# ── Build & train model ───────────────────────────────────────────────────────
model = NeuralNetworkRegressor(input_dim=X_train_s.shape[1], output_dim=3, learning_rate=LR)

train_losses, val_losses = [], []
n_batches = len(X_train_s) // BATCH_SIZE

print(f"Training {EPOCHS} epochs | {n_batches} batches/epoch | LR={LR}")
for epoch in range(1, EPOCHS + 1):
    # Shuffle
    idx = np.random.permutation(len(X_train_s))
    X_shuf, y_shuf = X_train_s[idx], y_train_s[idx]
    
    epoch_loss = 0.0
    for b in range(n_batches):
        X_batch = X_shuf[b*BATCH_SIZE:(b+1)*BATCH_SIZE]
        y_batch = y_shuf[b*BATCH_SIZE:(b+1)*BATCH_SIZE]
        epoch_loss += model.train_step(X_batch, y_batch)
    
    train_losses.append(epoch_loss / n_batches)
    
    # Validation loss (on original scale)
    y_pred_s = model.predict(X_val_s)
    y_pred   = scaler_y.inverse_transform(y_pred_s)
    val_loss = mean_squared_error(y_val, y_pred)
    val_losses.append(val_loss)
    
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Train MSE: {train_losses[-1]:.4f} | Val MSE: {val_loss:.2f}")

# ── Evaluate ─────────────────────────────────────────────────────────────────
y_pred_final = scaler_y.inverse_transform(model.predict(X_val_s))

results = {}
for i, target in enumerate(TARGETS):
    rmse = np.sqrt(mean_squared_error(y_val[:, i], y_pred_final[:, i]))
    mae  = mean_absolute_error(y_val[:, i], y_pred_final[:, i])
    r2   = r2_score(y_val[:, i], y_pred_final[:, i])
    results[target] = {'RMSE': round(rmse, 4), 'MAE': round(mae, 4), 'R2': round(r2, 4)}
    print(f"{target:15s} → RMSE: {rmse:.2f} | MAE: {mae:.2f} | R²: {r2:.4f}")

# ── Save artifacts ────────────────────────────────────────────────────────────
# Save model weights as numpy arrays
weights = {
    f'layer_{i}_W': layer.W for i, layer in enumerate(model.layers)
}
weights.update({
    f'layer_{i}_b': layer.b for i, layer in enumerate(model.layers)
})
np.savez('outputs/models/nn_weights.npz', **weights)

# Save loss curves
np.save('outputs/models/train_losses.npy', np.array(train_losses))
np.save('outputs/models/val_losses.npy',   np.array(val_losses))

# Save eval results
with open('outputs/models/eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)

with open('outputs/models/feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

print("\nAll artifacts saved to outputs/models/")

Training 200 epochs | 76 batches/epoch | LR=0.001
Epoch   1 | Train MSE: 0.7819 | Val MSE: 3094.91
Epoch  20 | Train MSE: 0.4456 | Val MSE: 3289.04
Epoch  40 | Train MSE: 0.3561 | Val MSE: 3323.79
Epoch  60 | Train MSE: 0.2921 | Val MSE: 3479.52
Epoch  80 | Train MSE: 0.2443 | Val MSE: 3838.53
Epoch 100 | Train MSE: 0.2117 | Val MSE: 3825.30
Epoch 120 | Train MSE: 0.1911 | Val MSE: 4036.17
Epoch 140 | Train MSE: 0.1839 | Val MSE: 4222.25
Epoch 160 | Train MSE: 0.1534 | Val MSE: 4456.69
Epoch 180 | Train MSE: 0.1577 | Val MSE: 4438.24
Epoch 200 | Train MSE: 0.1367 | Val MSE: 4683.40
avg_d_mbps      → RMSE: 113.93 | MAE: 75.08 | R²: 0.1445
avg_u_mbps      → RMSE: 14.09 | MAE: 9.99 | R²: -0.2911
avg_lat_ms      → RMSE: 29.53 | MAE: 12.94 | R²: -0.4178

All artifacts saved to outputs/models/


In [32]:
"""
train.py  –  Fixed Training Script
====================================
Fixes vs v1:
  1. RANDOM shuffle split instead of temporal — your data ordering caused
     train/test to be from completely different time periods / distributions.
     Use stratified random split so both sets see all areas and years.
  2. Separate StandardScaler per target — download (~100 Mbps) and latency
     (~30 ms) have very different magnitudes; sharing one scaler distorts both.
  3. Train/Val loss both reported on the SAME scale (normalised MSE) so you
     can actually compare them and detect overfitting early.
  4. Early stopping with patience — stops when val loss stops improving.
  5. Learning rate reduced to 3e-4 (Adam is sensitive on small tabular data).
  6. Smaller batch size (32) for better gradient estimates on ~4 k rows.
"""
 
import numpy as np
import pandas as pd
import pickle, json, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
 
# ── make sure Python can find architecture.py ────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('/scripts/architecture')))
from architecture import NeuralNetworkRegressor
 
# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────
EPOCHS       = 300
BATCH_SIZE   = 32
LR           = 3e-4
L2_LAMBDA    = 1e-4
PATIENCE     = 30        # early stopping
RANDOM_SEED  = 42
np.random.seed(RANDOM_SEED)
 
FEATURE_COLS = [
    'latitude', 'longitude',
    'nearest_tower_distance_km',
    'tower_count_1km', 'tower_count_2km', 'tower_count_5km',
    'tower_density_ratio',
    'digital_elevation_model',
    'region_enc', 'typeOfArea_enc', 'city_enc',
    'demand_growth_pct',
    'year_trend', 'quarter',
    'area_rolling_d_mbps',
    'tests'
]
TARGETS = ['avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms']
 
# ─────────────────────────────────────────────────────────────────────────────
# Load & split
# ─────────────────────────────────────────────────────────────────────────────
print("Loading data …")
features = pd.read_csv('data/processed/features_engineered.csv')
 
# Drop any rows with NaN in feature or target columns
features = features.dropna(subset=FEATURE_COLS + TARGETS).reset_index(drop=True)
 
X = features[FEATURE_COLS].values.astype(float)
y = features[TARGETS].values.astype(float)
 
print(f"Dataset size: {X.shape[0]} rows × {X.shape[1]} features")
 
# ── FIX 1: random stratified-ish split (not temporal) ────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, shuffle=True
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
 
# ─────────────────────────────────────────────────────────────────────────────
# Scale
# ─────────────────────────────────────────────────────────────────────────────
os.makedirs('outputs/models', exist_ok=True)
os.makedirs('assets/figures', exist_ok=True)
 
# Feature scaler
scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s  = scaler_X.transform(X_test)
 
# ── FIX 2: one scaler per target ─────────────────────────────────────────────
scalers_y = {}
y_train_s = np.zeros_like(y_train)
y_test_s  = np.zeros_like(y_test)
 
for i, target in enumerate(TARGETS):
    sc = StandardScaler()
    y_train_s[:, i] = sc.fit_transform(y_train[:, i].reshape(-1, 1)).ravel()
    y_test_s[:, i]  = sc.transform(y_test[:, i].reshape(-1, 1)).ravel()
    scalers_y[target] = sc
 
with open('outputs/models/scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)
with open('outputs/models/scalers_y.pkl', 'wb') as f:
    pickle.dump(scalers_y, f)
print("Scalers saved.")
 
# ─────────────────────────────────────────────────────────────────────────────
# Build model
# ─────────────────────────────────────────────────────────────────────────────
model = NeuralNetworkRegressor(
    input_dim=X_train_s.shape[1],
    learning_rate=LR,
    l2_lambda=L2_LAMBDA
)
print(f"\nModel ready. Starting training …\n{'─'*60}")
 
# ─────────────────────────────────────────────────────────────────────────────
# Training loop with early stopping
# ─────────────────────────────────────────────────────────────────────────────
train_losses, val_losses = [], []
best_val_loss  = np.inf
best_weights   = None
patience_count = 0
 
n_batches = max(1, len(X_train_s) // BATCH_SIZE)
 
for epoch in range(1, EPOCHS + 1):
    # Shuffle training data each epoch
    idx = np.random.permutation(len(X_train_s))
    X_sh, y_sh = X_train_s[idx], y_train_s[idx]
 
    epoch_loss = 0.0
    for b in range(n_batches):
        Xb = X_sh[b * BATCH_SIZE: (b + 1) * BATCH_SIZE]
        yb = y_sh[b * BATCH_SIZE: (b + 1) * BATCH_SIZE]
        epoch_loss += model.train_step(Xb, yb)
    train_mse = epoch_loss / n_batches
 
    # ── FIX 3: val loss on same normalised scale ──────────────────────────
    y_pred_s = model.predict(X_test_s)            # normalised predictions
    val_mse  = np.mean((y_pred_s - y_test_s) ** 2)
 
    train_losses.append(train_mse)
    val_losses.append(val_mse)
 
    # ── FIX 4: early stopping ─────────────────────────────────────────────
    if val_mse < best_val_loss - 1e-6:
        best_val_loss  = val_mse
        best_weights   = model.get_weights()
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"\n⏹  Early stopping at epoch {epoch} "
                  f"(no improvement for {PATIENCE} epochs)")
            break
 
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | "
              f"Train MSE (norm): {train_mse:.4f} | "
              f"Val MSE (norm):   {val_mse:.4f}")
 
# Restore best weights
model.load_weights(best_weights)
print(f"\n✅ Best val MSE (norm): {best_val_loss:.4f}")
 
# ─────────────────────────────────────────────────────────────────────────────
# Evaluate on original scale
# ─────────────────────────────────────────────────────────────────────────────
y_pred_s_final = model.predict(X_test_s)
 
# Inverse-transform each target separately
y_pred_orig = np.zeros_like(y_test)
for i, target in enumerate(TARGETS):
    y_pred_orig[:, i] = (
        scalers_y[target]
        .inverse_transform(y_pred_s_final[:, i].reshape(-1, 1))
        .ravel()
    )
 
print(f"\n{'─'*60}")
print(f"{'Target':<18} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
print(f"{'─'*60}")
results = {}
for i, target in enumerate(TARGETS):
    rmse = np.sqrt(mean_squared_error(y_test[:, i], y_pred_orig[:, i]))
    mae  = mean_absolute_error(y_test[:, i], y_pred_orig[:, i])
    r2   = r2_score(y_test[:, i], y_pred_orig[:, i])
    results[target] = {'RMSE': round(rmse, 3),
                       'MAE':  round(mae,  3),
                       'R2':   round(r2,   4)}
    print(f"{target:<18} {rmse:>8.2f} {mae:>8.2f} {r2:>8.4f}")
print(f"{'─'*60}")
 
# ─────────────────────────────────────────────────────────────────────────────
# Save artifacts
# ─────────────────────────────────────────────────────────────────────────────
np.savez('outputs/models/nn_weights.npz', **model.get_weights())
np.save('outputs/models/train_losses.npy', np.array(train_losses))
np.save('outputs/models/val_losses.npy',   np.array(val_losses))
 
with open('outputs/models/eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)
with open('outputs/models/feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)
 
print("\nAll artifacts saved → outputs/models/")
 
# ─────────────────────────────────────────────────────────────────────────────
# Loss curve plot (saved for evaluation notebook)
# ─────────────────────────────────────────────────────────────────────────────
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 4))
    plt.plot(train_losses, label='Train MSE (normalised)', linewidth=1.5)
    plt.plot(val_losses,   label='Val MSE (normalised)',   linewidth=1.5, alpha=0.8)
    plt.xlabel('Epoch')
    plt.ylabel('MSE (normalised scale)')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig('assets/figures/loss_curves.png', dpi=150)
    plt.close()
    print("Loss curve saved → assets/figures/loss_curves.png")
except Exception as e:
    print(f"(Could not save plot: {e})")

Loading data …
Dataset size: 6086 rows × 16 features
Train: 4868 | Test: 1218
Scalers saved.

Model ready. Starting training …
────────────────────────────────────────────────────────────
Epoch   1 | Train MSE (norm): 1.3040 | Val MSE (norm):   1.1764
Epoch  20 | Train MSE (norm): 0.6450 | Val MSE (norm):   0.8379
Epoch  40 | Train MSE (norm): 0.6024 | Val MSE (norm):   0.7977
Epoch  60 | Train MSE (norm): 0.5588 | Val MSE (norm):   0.7958
Epoch  80 | Train MSE (norm): 0.5518 | Val MSE (norm):   0.7593
Epoch 100 | Train MSE (norm): 0.5304 | Val MSE (norm):   0.8065
Epoch 120 | Train MSE (norm): 0.5022 | Val MSE (norm):   0.7587
Epoch 140 | Train MSE (norm): 0.4915 | Val MSE (norm):   0.7536

⏹  Early stopping at epoch 147 (no improvement for 30 epochs)

✅ Best val MSE (norm): 0.7249

────────────────────────────────────────────────────────────
Target                 RMSE      MAE       R²
────────────────────────────────────────────────────────────
avg_d_mbps            83.63    53.22 

In [7]:
import pandas as pd
speed_df = pd.read_csv('data/speedtests.csv')
speed_df.columns = speed_df.columns.str.strip()
speed_df['date'] = pd.to_datetime(speed_df['date'], format='%m/%d/%Y', errors='coerce')
speed_df['year'] = speed_df['date'].dt.year

area_df = speed_df[speed_df['area'] == speed_df['area'].iloc[0]]
print("area name:", speed_df['area'].iloc[0])
print("avg_d_mbps sample values:", area_df['avg_d_mbps'].head(10).tolist())
print("median:", area_df['avg_d_mbps'].median())
print("mean:", area_df['avg_d_mbps'].mean())
print("year min:", speed_df['year'].min())

area name: Jazirat as Sayah
avg_d_mbps sample values: [4.029, 23.765, 29.366, 27.02, 54.6, 66.72, 47.023, 13.069, 3.734, 19.806]
median: 107.567
mean: 150.04008429118772
year min: 2020


In [8]:
import json
with open('assets/encoders.json') as f:
    enc = json.load(f)

print("Keys in encoders:", list(enc.keys()))
print("Sample region keys:", list(enc.get('region', {}).keys())[:5])
print("Sample city keys:", list(enc.get('city', {}).keys())[:5])
print("Sample typeOfArea keys:", list(enc.get('typeOfArea', {}).keys())[:5])

Keys in encoders: ['region', 'typeOfArea', 'city']
Sample region keys: ['Capital Governorate', 'Muharraq Governorate', 'Northern Governorate', 'Southern Governorate']
Sample city keys: ["A'ali", 'Abu Baham', 'Abu Saiba', 'Abu Saybi`', 'Ad Dayr']
Sample typeOfArea keys: ['HTL', 'ISL', 'ISLS', 'ISLX', 'PPL']


In [9]:
import pandas as pd
import joblib
import json

# Load model and features
with open('outputs/models/feature_cols2.json') as f:
    feat_cols = json.load(f)

models = joblib.load('outputs/models/gb_models.pkl')

# Load a real row from training data
df = pd.read_csv('data/processed/features_engineered.csv')
sample = df[feat_cols].iloc[0:1]

print("Sample input row:")
print(sample.to_dict('records')[0])
print()
for target in ['avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms']:
    pred = models[target].predict(sample)[0]
    actual = df[target].iloc[0]
    print(f"{target}: predicted={pred:.2f}, actual={actual:.2f}")

Sample input row:
{'latitude': 26.16776393, 'longitude': 50.516784668, 'nearest_tower_distance_km': 0.910574059881922, 'tower_count_1km': 3.2, 'tower_count_2km': 10.6, 'tower_count_5km': 184.0, 'tower_density_ratio': 0.0172972972972972, 'digital_elevation_model': 10.0, 'region_enc': 2, 'typeOfArea_enc': 5, 'city_enc': 0, 'demand_growth_pct': 0.0, 'year_trend': 0, 'quarter': 1, 'area_rolling_d_mbps': 152.42066227997978, 'tests': 52, 'area_median_d': 178.8578333333333, 'area_median_u': 23.14375, 'area_median_lat': 20.416666666666668, 'area_test_count': 2070, 'distance_x_density': 0.0049220219453076}

avg_d_mbps: predicted=3.73, actual=34.61
avg_u_mbps: predicted=2.69, actual=6.63
avg_lat_ms: predicted=3.13, actual=21.20


In [22]:
import inspect
sys.path.insert(0, os.path.dirname(os.path.abspath('/scripts/architecture')))
from architecture import NeuralNetworkRegressor

In [25]:
print(inspect.signature(NeuralNetworkRegressor.__init__))

(self, input_dim, learning_rate=0.0003, l2_lambda=0.0001)


In [24]:
import importlib, sys

# Remove any cached version
for key in list(sys.modules.keys()):
    if 'architecture' in key:
        del sys.modules[key]

import architecture
importlib.reload(architecture)
from architecture import NeuralNetworkRegressor

In [29]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train_s, y_train)
y_dummy = dummy.predict(X_test_s)

for i, target in enumerate(TARGETS):
    r2 = r2_score(y_test[:, i], y_dummy[:, i])
    print(f"{target}: Dummy R² = {r2:.4f}")  # will be 0.0 by definition

avg_d_mbps: Dummy R² = -0.0064
avg_u_mbps: Dummy R² = -0.0032
avg_lat_ms: Dummy R² = -0.0005


In [37]:
"""
train.py  –  Fixed Training Script
====================================
Fixes vs v1:
  1. RANDOM shuffle split instead of temporal
  2. Separate StandardScaler per target
  3. Train/Val loss both on the SAME normalised scale
  4. Early stopping with patience
  5. Learning rate 3e-4, batch size 32
  6. Proper train / val / test three-way split
"""
 
import numpy as np
import pandas as pd
import pickle, json, os, sys
 
sys.path.insert(0, os.path.dirname(os.path.abspath('/scripts/architecture')))
from architecture import NeuralNetworkRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
 
# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────
EPOCHS      = 300
BATCH_SIZE  = 32
LR          = 3e-4
L2_LAMBDA   = 1e-4
PATIENCE    = 30
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
 
FEATURE_COLS = [
    'latitude', 'longitude',
    'nearest_tower_distance_km',
    'tower_count_1km', 'tower_count_2km', 'tower_count_5km',
    'tower_density_ratio',
    'digital_elevation_model',
    'region_enc', 'typeOfArea_enc', 'city_enc',
    'demand_growth_pct',
    'year_trend', 'quarter',
    'area_rolling_d_mbps',
    'tests'
]
TARGETS = ['avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms']
 
# ─────────────────────────────────────────────────────────────────────────────
# Load & split
# ─────────────────────────────────────────────────────────────────────────────
print("Loading data ...")
features = pd.read_csv('data/processed/features_engineered.csv')
features = features.dropna(subset=FEATURE_COLS + TARGETS).reset_index(drop=True)
 
X = features[FEATURE_COLS].values.astype(float)
y = features[TARGETS].values.astype(float)
print(f"Dataset size: {X.shape[0]} rows x {X.shape[1]} features")
 
# Three-way split: 70% train | 15% val (early stopping) | 15% test (final eval)
X_train_val, X_test,  y_train_val, y_test  = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_SEED, shuffle=True)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, random_state=RANDOM_SEED, shuffle=True)
 
print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
 
# ─────────────────────────────────────────────────────────────────────────────
# Scale
# ─────────────────────────────────────────────────────────────────────────────
os.makedirs('outputs/models', exist_ok=True)
os.makedirs('assets/figures', exist_ok=True)
 
scaler_X  = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_val_s   = scaler_X.transform(X_val)
X_test_s  = scaler_X.transform(X_test)      # FIX: was missing entirely
 
scalers_y = {}
y_train_s = np.zeros_like(y_train, dtype=float)
y_val_s   = np.zeros_like(y_val,   dtype=float)
 
for i, target in enumerate(TARGETS):
    sc = StandardScaler()
    y_train_s[:, i] = sc.fit_transform(y_train[:, i].reshape(-1, 1)).ravel()
    y_val_s[:, i]   = sc.transform(y_val[:, i].reshape(-1, 1)).ravel()  # FIX: was y_test
    scalers_y[target] = sc
 
with open('outputs/models/scaler_X.pkl',  'wb') as f: pickle.dump(scaler_X,  f)
with open('outputs/models/scalers_y.pkl', 'wb') as f: pickle.dump(scalers_y, f)
print("Scalers saved.")
 
# ─────────────────────────────────────────────────────────────────────────────
# Build model
# ─────────────────────────────────────────────────────────────────────────────
model = NeuralNetworkRegressor(
    input_dim=X_train_s.shape[1],
    learning_rate=LR,
    l2_lambda=L2_LAMBDA
)
print(f"\nModel ready. Starting training ...\n{'--'*30}")
 
# ─────────────────────────────────────────────────────────────────────────────
# Training loop with early stopping
# ─────────────────────────────────────────────────────────────────────────────
train_losses, val_losses = [], []
best_val_loss  = np.inf
best_weights   = None
patience_count = 0
n_batches      = max(1, len(X_train_s) // BATCH_SIZE)
 
for epoch in range(1, EPOCHS + 1):
    idx = np.random.permutation(len(X_train_s))
    X_sh, y_sh = X_train_s[idx], y_train_s[idx]
 
    epoch_loss = 0.0
    for b in range(n_batches):
        Xb = X_sh[b * BATCH_SIZE: (b + 1) * BATCH_SIZE]
        yb = y_sh[b * BATCH_SIZE: (b + 1) * BATCH_SIZE]
        epoch_loss += model.train_step(Xb, yb)
    train_mse = epoch_loss / n_batches
 
    y_pred_val_s = model.predict(X_val_s)
    val_mse      = np.mean((y_pred_val_s - y_val_s) ** 2)
 
    train_losses.append(train_mse)
    val_losses.append(val_mse)
 
    if val_mse < best_val_loss - 1e-6:
        best_val_loss  = val_mse
        best_weights   = model.get_weights()
        patience_count = 0
    
 
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | "
              f"Train MSE (norm): {train_mse:.4f} | "
              f"Val MSE   (norm): {val_mse:.4f}")
 
model.load_weights(best_weights)
print(f"\nBest val MSE (norm): {best_val_loss:.4f}")
 
# ─────────────────────────────────────────────────────────────────────────────
# Final evaluation on held-out TEST set (original scale)
# ─────────────────────────────────────────────────────────────────────────────
y_pred_s_final = model.predict(X_test_s)
y_pred_orig    = np.zeros_like(y_test, dtype=float)
 
for i, target in enumerate(TARGETS):
    y_pred_orig[:, i] = (
        scalers_y[target]
        .inverse_transform(y_pred_s_final[:, i].reshape(-1, 1))
        .ravel()
    )
 
print(f"\n{'--'*30}")
print(f"{'Target':<18} {'RMSE':>8} {'MAE':>8} {'R2':>8}")
print(f"{'--'*30}")
results = {}
for i, target in enumerate(TARGETS):
    rmse = np.sqrt(mean_squared_error(y_test[:, i], y_pred_orig[:, i]))
    mae  = mean_absolute_error(y_test[:, i], y_pred_orig[:, i])
    r2   = r2_score(y_test[:, i], y_pred_orig[:, i])
    results[target] = {'RMSE': round(rmse, 3), 'MAE': round(mae, 3), 'R2': round(r2, 4)}
    print(f"{target:<18} {rmse:>8.2f} {mae:>8.2f} {r2:>8.4f}")
print(f"{'--'*30}")
 
# ─────────────────────────────────────────────────────────────────────────────
# Save artifacts
# ─────────────────────────────────────────────────────────────────────────────
np.savez('outputs/models/nn_weights.npz', **model.get_weights())
np.save('outputs/models/train_losses.npy', np.array(train_losses))
np.save('outputs/models/val_losses.npy',   np.array(val_losses))
 
with open('outputs/models/eval_results.json', 'w') as f: json.dump(results,      f, indent=2)
with open('outputs/models/feature_cols.json', 'w') as f: json.dump(FEATURE_COLS, f, indent=2)
 
print("\nAll artifacts saved -> outputs/models/")
 
# ─────────────────────────────────────────────────────────────────────────────
# Loss curve
# ─────────────────────────────────────────────────────────────────────────────
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 4))
    plt.plot(train_losses, label='Train MSE (normalised)', linewidth=1.5)
    plt.plot(val_losses,   label='Val MSE (normalised)',   linewidth=1.5, alpha=0.8)
    plt.xlabel('Epoch')
    plt.ylabel('MSE (normalised scale)')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig('assets/figures/loss_curves.png', dpi=150)
    plt.close()
    print("Loss curve saved -> assets/figures/loss_curves.png")
except Exception as e:
    print(f"(Could not save plot: {e})")

Loading data ...
Dataset size: 6086 rows x 16 features
Train: 4397 | Val: 776 | Test: 913
Scalers saved.

Model ready. Starting training ...
------------------------------------------------------------
Epoch   1 | Train MSE (norm): 1.3137 | Val MSE   (norm): 1.2018
Epoch  20 | Train MSE (norm): 0.6597 | Val MSE   (norm): 0.9049
Epoch  40 | Train MSE (norm): 0.6065 | Val MSE   (norm): 0.8765
Epoch  60 | Train MSE (norm): 0.5576 | Val MSE   (norm): 0.8460
Epoch  80 | Train MSE (norm): 0.5486 | Val MSE   (norm): 0.8422
Epoch 100 | Train MSE (norm): 0.5148 | Val MSE   (norm): 0.8535
Epoch 120 | Train MSE (norm): 0.5140 | Val MSE   (norm): 0.8285
Epoch 140 | Train MSE (norm): 0.5014 | Val MSE   (norm): 0.8463
Epoch 160 | Train MSE (norm): 0.4728 | Val MSE   (norm): 0.8780
Epoch 180 | Train MSE (norm): 0.4806 | Val MSE   (norm): 0.8647
Epoch 200 | Train MSE (norm): 0.4515 | Val MSE   (norm): 0.8364
Epoch 220 | Train MSE (norm): 0.4504 | Val MSE   (norm): 0.8441
Epoch 240 | Train MSE (norm): 

In [38]:
#Evaluation

In [59]:
"""
04_evaluation.py  –  Model Evaluation & Baseline Comparison
=============================================================
Run this after train.py has completed and saved artifacts.
 
Models evaluated:
  1. Dummy (mean predictor)      — the absolute floor; any useful model beats this
  2. Gradient Boosting           — strong classical baseline (sklearn)
  3. Random Forest               — NEW: ensemble of decision trees, often best at
                                   tabular data, highly interpretable feature importance
  4. Neural Network (ours)       — custom NumPy architecture from scratch
 
Metrics reported per model per target:
  • RMSE  – Root Mean Squared Error        (penalises large mistakes more)
  • MAE   – Mean Absolute Error            (average size of any mistake in units)
  • MAPE  – Mean Absolute Percentage Error (how wrong in %, key for stakeholders)
  • R²    – Coefficient of determination  (variance explained; 1.0 = perfect)
 
At the end, the script runs an automatic winner-selection algorithm that
scores all non-dummy models across all targets and all metrics, then prints
a clear recommendation with a full justification.
 
WHY add Random Forest?
  Random Forest averages many independent decision trees, which:
    • reduces the variance that single trees suffer from
    • handles non-linear feature interactions naturally
    • is very sample-efficient (works well on our ~6,000 row dataset)
    • provides direct feature importances (useful for the presentation)
  On tabular regression problems of this size, RF often outperforms both
  Gradient Boosting and neural networks.  Adding it lets us make an honest,
  evidence-based claim about which model to deploy.
"""
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pickle, json, sys, os
 
sys.path.insert(0, os.path.dirname(os.path.abspath('scripts/architecture.py')))
from architecture import NeuralNetworkRegressor
from sklearn.model_selection   import train_test_split
from sklearn.metrics           import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble          import GradientBoostingRegressor, RandomForestRegressor
from sklearn.dummy             import DummyRegressor
 
os.makedirs('assets/figures', exist_ok=True)
 
TARGETS      = ['avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms']
TARGET_UNITS = {'avg_d_mbps': 'Mbps', 'avg_u_mbps': 'Mbps', 'avg_lat_ms': 'ms'}
RANDOM_SEED  = 42
 
# ─────────────────────────────────────────────────────────────────────────────
# Metric helpers
# ─────────────────────────────────────────────────────────────────────────────
 
def mape(y_true, y_pred, eps=1e-8):
    """
    Mean Absolute Percentage Error in %.
 
    Formula:  (1/n) × Σ |actual − predicted| / |actual| × 100
 
    Guard: rows where |actual| < eps are excluded to avoid division-by-zero
    on failed tests that recorded 0 Mbps (rare but present in dataset).
    Without this guard, a single zero-speed row would inflate MAPE to infinity.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid  = np.abs(y_true) > eps
    if valid.sum() == 0:
        return float('nan')
    return float(
        np.mean(np.abs(y_true[valid] - y_pred[valid]) / np.abs(y_true[valid])) * 100
    )
 
 
def all_metrics(y_true, y_pred):
    """Return a dict of all four metrics for one target column."""
    return {
        'RMSE': round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 3),
        'MAE':  round(float(mean_absolute_error(y_true, y_pred)),         3),
        'MAPE': round(mape(y_true, y_pred),                               1),
        'R2':   round(float(r2_score(y_true, y_pred)),                    4),
    }
 
 
def rank_models(results_dict, target, lower_better_metrics=('RMSE','MAE','MAPE'),
                higher_better_metrics=('R2',)):
    """
    Rank models for a given target by sorting each metric independently,
    then summing the rank positions.  Lower total rank = better overall.
 
    Returns a list of (model_name, total_rank_score, per_metric_ranks) sorted
    from best (lowest score) to worst.
    """
    model_names = list(results_dict.keys())
    rank_scores = {name: 0 for name in model_names}
    per_metric  = {name: {} for name in model_names}
 
    for metric in lower_better_metrics + higher_better_metrics:
        values = [(name, results_dict[name][target][metric]) for name in model_names]
        if metric in lower_better_metrics:
            sorted_vals = sorted(values, key=lambda x: x[1])         # ascending
        else:
            sorted_vals = sorted(values, key=lambda x: x[1], reverse=True)  # descending
 
        for rank_pos, (name, val) in enumerate(sorted_vals, start=1):
            rank_scores[name]          += rank_pos
            per_metric[name][metric]    = rank_pos
 
    ranked = sorted(rank_scores.items(), key=lambda x: x[1])
    return [(name, score, per_metric[name]) for name, score in ranked]
 
 
# ─────────────────────────────────────────────────────────────────────────────
# Load artifacts
# ─────────────────────────────────────────────────────────────────────────────
with open('outputs/models/scaler_X.pkl',  'rb') as f: scaler_X     = pickle.load(f)
with open('outputs/models/scalers_y.pkl', 'rb') as f: scalers_y    = pickle.load(f)
with open('outputs/models/feature_cols.json')   as f: FEATURE_COLS = json.load(f)
 
# ─────────────────────────────────────────────────────────────────────────────
# Recreate exact splits used in train.py
# Same random_state + test_size = guaranteed identical test rows
# ─────────────────────────────────────────────────────────────────────────────
features = pd.read_csv('data/processed/features_engineered.csv')
features = features.dropna(subset=FEATURE_COLS + TARGETS).reset_index(drop=True)
 
X = features[FEATURE_COLS].values.astype(float)
y = features[TARGETS].values.astype(float)
 
X_train_val, X_test,  y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_SEED, shuffle=True)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, random_state=RANDOM_SEED, shuffle=True)
 
X_train_s = scaler_X.transform(X_train)
X_test_s  = scaler_X.transform(X_test)
 
print(f"Train: {X_train.shape[0]} rows  |  Val: {X_val.shape[0]} rows  "
      f"|  Test: {X_test.shape[0]} rows")
 
# ─────────────────────────────────────────────────────────────────────────────
# MODEL 1 – Dummy mean predictor (baseline floor)
# ─────────────────────────────────────────────────────────────────────────────
print("\nFitting Dummy predictor …")
dummy_preds   = np.zeros_like(y_test)
for i, target in enumerate(TARGETS):
    d = DummyRegressor(strategy='mean')
    d.fit(X_train_s, y_train[:, i])
    dummy_preds[:, i] = d.predict(X_test_s)
 
# ─────────────────────────────────────────────────────────────────────────────
# MODEL 2 – Gradient Boosting
# Hyperparameters: 200 trees (more than default 100 for stability),
# learning_rate=0.05 (slower learning → less overfit), max_depth=4
# ─────────────────────────────────────────────────────────────────────────────
print("Fitting Gradient Boosting (200 trees × 3 targets) …")
gb_preds  = np.zeros_like(y_test)
gb_models = {}
for i, target in enumerate(TARGETS):
    gb = GradientBoostingRegressor(
        n_estimators  = 200,
        learning_rate = 0.05,
        max_depth     = 4,
        random_state  = RANDOM_SEED,
    )
    gb.fit(X_train_s, y_train[:, i])
    gb_models[target]  = gb
    gb_preds[:, i]     = gb.predict(X_test_s)
    print(f"  GB [{target}] done")
 
# ─────────────────────────────────────────────────────────────────────────────
# MODEL 3 – Random Forest  ← NEW
#
# WHY these hyperparameters:
#   n_estimators=300  → 300 trees gives a stable ensemble; more trees rarely hurt
#   max_depth=None    → trees grow until all leaves are pure; let the data decide
#   min_samples_leaf=4 → each leaf needs ≥4 samples to avoid memorising noise
#   max_features='sqrt' → each split considers √(n_features) candidates, which
#                         forces tree diversity (key to RF's variance reduction)
#   n_jobs=-1         → uses all CPU cores; RF is embarrassingly parallel
# ─────────────────────────────────────────────────────────────────────────────
print("Fitting Random Forest (300 trees × 3 targets) …")
rf_preds  = np.zeros_like(y_test)
rf_models = {}
for i, target in enumerate(TARGETS):
    rf = RandomForestRegressor(
        n_estimators    = 300,
        max_depth       = None,          # grow full trees
        min_samples_leaf= 4,             # regularisation
        max_features    = 'sqrt',        # feature diversity per split
        random_state    = RANDOM_SEED,
        n_jobs          = -1,
    )
    rf.fit(X_train_s, y_train[:, i])
    rf_models[target] = rf
    rf_preds[:, i]    = rf.predict(X_test_s)
    print(f"  RF [{target}] done")
 
# ─────────────────────────────────────────────────────────────────────────────
# MODEL 4 – Neural Network (custom NumPy, from scratch)
# ─────────────────────────────────────────────────────────────────────────────
print("Loading Neural Network weights …")
weights   = np.load('outputs/models/nn_weights.npz')
nn_model  = NeuralNetworkRegressor(input_dim=X_test_s.shape[1])
nn_model.load_weights(weights)
 
y_pred_s  = nn_model.predict(X_test_s)
nn_preds  = np.zeros_like(y_test)
for i, target in enumerate(TARGETS):
    nn_preds[:, i] = (
        scalers_y[target]
        .inverse_transform(y_pred_s[:, i].reshape(-1, 1))
        .ravel()
    )
 
# ─────────────────────────────────────────────────────────────────────────────
# Compute all metrics for all models
# ─────────────────────────────────────────────────────────────────────────────
print("\nComputing metrics …")
 
all_results = {
    'Dummy (mean)':         {t: all_metrics(y_test[:, i], dummy_preds[:, i]) for i, t in enumerate(TARGETS)},
    'Gradient Boosting':    {t: all_metrics(y_test[:, i], gb_preds[:, i])    for i, t in enumerate(TARGETS)},
    'Random Forest':        {t: all_metrics(y_test[:, i], rf_preds[:, i])    for i, t in enumerate(TARGETS)},
    'Neural Network (ours)':{t: all_metrics(y_test[:, i], nn_preds[:, i])    for i, t in enumerate(TARGETS)},
}
 
# Shorthand refs for later use
dummy_results = all_results['Dummy (mean)']
gb_results    = all_results['Gradient Boosting']
rf_results    = all_results['Random Forest']
nn_results    = all_results['Neural Network (ours)']
 
# ─────────────────────────────────────────────────────────────────────────────
# PRINT: Full comparison table
# ─────────────────────────────────────────────────────────────────────────────
W = 85
print(f"\n{'═'*W}")
print(f"{'Target':<18} {'Model':<28} {'RMSE':>8} {'MAE':>8} {'MAPE':>8} {'R²':>8}")
print(f"{'═'*W}")
 
model_order = list(all_results.keys())
for target in TARGETS:
    for name in model_order:
        r = all_results[name][target]
        print(f"{target:<18} {name:<28} "
              f"{r['RMSE']:>8.3f} {r['MAE']:>8.3f} "
              f"{r['MAPE']:>7.1f}% {r['R2']:>8.4f}")
    print(f"{'─'*W}")
 
# ─────────────────────────────────────────────────────────────────────────────
# AUTOMATIC WINNER SELECTION
#
# HOW IT WORKS:
#   For each target × metric, we rank the non-dummy models 1 (best) to 3 (worst).
#   We then sum the rank positions across all targets and all metrics.
#   The model with the lowest total rank score is the overall winner.
#
#   WHY rank-based rather than raw-score-based?
#   RMSE for download (~80 Mbps) and RMSE for latency (~34 ms) are on completely
#   different scales — you can't simply add them.  Rank normalises across scales
#   so each target × metric contributes equally to the final decision.
#
#   MAPE gets double weight (counted twice) because it is scale-independent
#   and the most meaningful metric for a non-technical stakeholder.
# ─────────────────────────────────────────────────────────────────────────────
 
# Only rank the real models (exclude Dummy — it's just the floor)
competing = {k: v for k, v in all_results.items() if k != 'Dummy (mean)'}
model_names = list(competing.keys())
 
METRICS_LOWER = ('RMSE', 'MAE', 'MAPE', 'MAPE')   # MAPE counted twice (double weight)
METRICS_HIGHER= ('R2',)
 
total_rank  = {name: 0 for name in model_names}
rank_detail = {name: [] for name in model_names}   # list of (target, metric, rank)
 
for target in TARGETS:
    for metric in METRICS_LOWER:
        ordered = sorted(model_names,
                         key=lambda n: competing[n][target][metric])  # ascending = lower is better
        for pos, name in enumerate(ordered, start=1):
            total_rank[name] += pos
            rank_detail[name].append((target, metric, pos))
 
    for metric in METRICS_HIGHER:
        ordered = sorted(model_names,
                         key=lambda n: competing[n][target][metric],
                         reverse=True)                                  # descending = higher is better
        for pos, name in enumerate(ordered, start=1):
            total_rank[name] += pos
            rank_detail[name].append((target, metric, pos))
 
# Sort: lowest rank score = best
ranked_models = sorted(total_rank.items(), key=lambda x: x[1])
winner_name, winner_score = ranked_models[0]
 
max_possible = len(model_names) * (len(TARGETS) * (len(METRICS_LOWER) + len(METRICS_HIGHER)))
min_possible = 1               * (len(TARGETS) * (len(METRICS_LOWER) + len(METRICS_HIGHER)))
 
print(f"\n{'━'*W}")
print(f"  🏆  AUTOMATIC WINNER SELECTION")
print(f"{'━'*W}")
print(f"  Ranking method: sum of per-(target × metric) rank positions")
print(f"  Metrics used  : RMSE, MAE, MAPE (×2, double-weighted), R²")
print(f"  Scale: {min_possible} (perfect) → {max_possible} (worst)\n")
 
for rank_pos, (name, score) in enumerate(ranked_models, start=1):
    medal = ['🥇', '🥈', '🥉'][rank_pos - 1] if rank_pos <= 3 else '  '
    r_d   = rf_results   if name == 'Random Forest'        else \
            gb_results   if name == 'Gradient Boosting'    else \
            nn_results
    arrow = '  ← BEST MODEL — use this for deployment' if rank_pos == 1 else ''
    print(f"  {medal} #{rank_pos}  {name:<28}  total rank score = {score}{arrow}")
 
print(f"\n{'─'*W}")
print(f"  WINNER: {winner_name}")
print(f"{'─'*W}")
 
# Print winner's metrics
w_res = competing[winner_name]
print(f"\n  {winner_name} metrics on held-out test set:")
print(f"  {'Target':<18} {'RMSE':>8} {'MAE':>8} {'MAPE':>8} {'R²':>8}")
for target in TARGETS:
    r = w_res[target]
    print(f"  {target:<18} {r['RMSE']:>8.3f} {r['MAE']:>8.3f} "
          f"{r['MAPE']:>7.1f}% {r['R2']:>8.4f}")
 
# Detailed justification
print(f"\n  WHY {winner_name} WINS:")
wins_per_metric = {m: [] for m in ('RMSE','MAE','MAPE','R2')}
for target in TARGETS:
    for metric in ('RMSE','MAE','MAPE','R2'):
        lower_better = metric != 'R2'
        best_val = min(competing[n][target][metric] for n in model_names) if lower_better \
                   else max(competing[n][target][metric] for n in model_names)
        if competing[winner_name][target][metric] == best_val:
            wins_per_metric[metric].append(target)
 
for metric, targets_won in wins_per_metric.items():
    if targets_won:
        print(f"    ✓ Best {metric} on: {', '.join(targets_won)}")
 
print(f"{'━'*W}\n")
 
# Save winner name for use by downstream scripts (app.py, predict.py)
with open('outputs/models/best_model_name.txt', 'w') as f:
    f.write(winner_name)
print(f"  Winner saved → outputs/models/best_model_name.txt")
 
# If RF or GB won, save that model for inference
if winner_name == 'Random Forest':
    import joblib
    joblib.dump(rf_models, 'outputs/models/best_model_sklearn.pkl')
    print(f"  RF models saved → outputs/models/best_model_sklearn.pkl")
elif winner_name == 'Gradient Boosting':
    import joblib
    joblib.dump(gb_models, 'outputs/models/best_model_sklearn.pkl')
    print(f"  GB models saved → outputs/models/best_model_sklearn.pkl")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1 – Loss curves (Neural Network only — GB/RF don't have epoch curves)
# ─────────────────────────────────────────────────────────────────────────────
train_losses = np.load('outputs/models/train_losses.npy')
val_losses   = np.load('outputs/models/val_losses.npy')
 
plt.figure(figsize=(9, 4))
plt.plot(train_losses, label='Train MSE (normalised)', linewidth=1.5, color='steelblue')
plt.plot(val_losses,   label='Val MSE (normalised)',   linewidth=1.5, color='tomato', alpha=0.85)
plt.xlabel('Epoch'); plt.ylabel('MSE (normalised scale)')
plt.title('Neural Network — Training vs Validation Loss Curves\n'
          '(confirms convergence — train and val loss both decrease)')
plt.legend(); plt.tight_layout()
plt.savefig('assets/figures/loss_curves.png', dpi=150)
print("\nSaved: assets/figures/loss_curves.png")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2 – Four-metric comparison: all 4 models side by side
# 2×2 grid: RMSE | MAE | MAPE | R²
# Dummy is included but greyed out so it's clear it's just the floor
# ─────────────────────────────────────────────────────────────────────────────
fig  = plt.figure(figsize=(16, 11))
gs   = gridspec.GridSpec(2, 2, hspace=0.50, wspace=0.38)
ax_list = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]
fig.suptitle('All Models — Full Metric Comparison\n'
             '(lower RMSE / MAE / MAPE = better  ·  higher R² = better  ·  🏆 = winner)',
             fontsize=13, fontweight='bold')
 
metric_specs = [
    ('RMSE', 'RMSE (original units)',   True,  'lower is better'),
    ('MAE',  'MAE (original units)',    True,  'lower is better'),
    ('MAPE', 'MAPE (%)',                True,  'lower % = more accurate relative to true value'),
    ('R2',   'R² (0–1)',                False, 'higher is better  |  R²=0 means no better than the mean'),
]
 
# Colours: grey=dummy, orange=GB, teal=RF, blue=NN; winner gets a ★ label
BAR_COLORS = {
    'Dummy (mean)':          '#aaaaaa',
    'Gradient Boosting':     '#f0ad4e',
    'Random Forest':         '#2a9d8f',
    'Neural Network (ours)': '#2196F3',
}
x = np.arange(len(TARGETS))
n_models = len(all_results)
total_w  = 0.8
width    = total_w / n_models
 
for ax, (metric, ylabel, lower_better, note) in zip(ax_list, metric_specs):
    for j, name in enumerate(model_order):
        vals = [all_results[name][t][metric] for t in TARGETS]
        offset = (j - n_models / 2 + 0.5) * width
        label  = f"{'🏆 ' if name == winner_name else ''}{name}"
        bars = ax.bar(x + offset, vals, width * 0.92,
                      label=label,
                      color=BAR_COLORS[name],
                      alpha=0.90 if name != 'Dummy (mean)' else 0.45,
                      edgecolor='white', linewidth=0.5)
        # Annotate bar tops with value
        for bar, val in zip(bars, vals):
            suffix = '%' if metric == 'MAPE' else ''
            fontsize = 6.5
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + (0.005 if metric == 'R2'
                                        else abs(bar.get_height()) * 0.04 + 0.15),
                    f'{val:.1f}{suffix}',
                    ha='center', va='bottom', fontsize=fontsize, fontweight='bold')
 
    ax.set_xticks(x)
    ax.set_xticklabels(TARGETS, rotation=12, ha='right', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{metric}  —  {note}', fontsize=9.5)
    ax.legend(fontsize=7.5, loc='upper right')
 
    if metric == 'R2':
        ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
        ax.text(x[-1] + total_w / 2 + 0.05, 0.008, 'Dummy floor',
                fontsize=7, color='gray', va='bottom')
 
plt.savefig('assets/figures/model_comparison_all_metrics.png', dpi=150, bbox_inches='tight')
print("Saved: assets/figures/model_comparison_all_metrics.png")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 3 – Rank scoreboard (visual summary of who won where)
# One heatmap-style grid: models × (target × metric), colour = rank position
# ─────────────────────────────────────────────────────────────────────────────
competing_names = list(competing.keys())
col_labels = [f"{t}\n{m}" for t in TARGETS for m in ('RMSE','MAE','MAPE','R2')]
rank_matrix = np.zeros((len(competing_names), len(col_labels)))
 
col_idx = 0
for target in TARGETS:
    for metric in ('RMSE', 'MAE', 'MAPE', 'R2'):
        lower_better = metric != 'R2'
        ordered = sorted(competing_names,
                         key=lambda n: competing[n][target][metric],
                         reverse=not lower_better)
        for pos, name in enumerate(ordered, start=1):
            row_idx = competing_names.index(name)
            rank_matrix[row_idx, col_idx] = pos
        col_idx += 1
 
fig, ax = plt.subplots(figsize=(14, 4))
cmap = plt.cm.RdYlGn_r   # red=worst rank, green=best rank (1st)
im   = ax.imshow(rank_matrix, cmap=cmap, aspect='auto', vmin=1, vmax=len(competing_names))
 
ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=8, rotation=30, ha='right')
ax.set_yticks(range(len(competing_names)))
ax.set_yticklabels(
    [f"{'🏆 ' if n == winner_name else '   '}{n}" for n in competing_names],
    fontsize=10
)
ax.set_title('Rank Scoreboard — Rank 1 (green) = best  ·  Rank 3 (red) = worst\n'
             f'🏆 Overall winner: {winner_name}', fontsize=11, fontweight='bold')
 
for r in range(len(competing_names)):
    for c in range(len(col_labels)):
        val = int(rank_matrix[r, c])
        ax.text(c, r, f'#{val}', ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if val == len(competing_names) else 'black')
 
plt.colorbar(im, ax=ax, label='Rank position (1=best)', shrink=0.8)
plt.tight_layout()
plt.savefig('assets/figures/rank_scoreboard.png', dpi=150, bbox_inches='tight')
print("Saved: assets/figures/rank_scoreboard.png")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 4 – Actual vs Predicted for the WINNER model (3 subplots)
# ─────────────────────────────────────────────────────────────────────────────
# Map winner name to its prediction array
winner_preds_map = {
    'Gradient Boosting':    gb_preds,
    'Random Forest':        rf_preds,
    'Neural Network (ours)':nn_preds,
}
winner_preds = winner_preds_map[winner_name]
 
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'🏆 Winner: {winner_name} — Predicted vs Actual (Test Set)', fontsize=13)
 
for i, (target, ax) in enumerate(zip(TARGETS, axes)):
    r    = competing[winner_name][target]
    unit = TARGET_UNITS[target]
    color = BAR_COLORS[winner_name]
    ax.scatter(y_test[:, i], winner_preds[:, i], alpha=0.30, s=8, color=color)
    lims = [min(y_test[:, i].min(), winner_preds[:, i].min()) * 0.95,
            max(y_test[:, i].max(), winner_preds[:, i].max()) * 1.05]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
    ax.set_xlabel(f'Actual ({unit})'); ax.set_ylabel(f'Predicted ({unit})')
    ax.set_title(f'{target}\nR²={r["R2"]}  |  MAPE={r["MAPE"]:.1f}%')
    ax.legend(fontsize=8)
 
plt.tight_layout()
plt.savefig('assets/figures/winner_actual_vs_predicted.png', dpi=150)
print("Saved: assets/figures/winner_actual_vs_predicted.png")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 5 – Feature importances (RF and GB both support this; show side by side)
# WHY: Feature importances prove the model learned sensible things.
#      "Year" and "nearest tower" should rank highly — if they don't,
#      investigate data leakage or encoding bugs.
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Importances — what each model uses to make predictions',
             fontsize=12, fontweight='bold')
 
for ax, (name, models_dict, color) in zip(axes, [
    ('Random Forest',     rf_models, '#2a9d8f'),
    ('Gradient Boosting', gb_models, '#f0ad4e'),
]):
    # Average importance across all 3 target models
    imp_matrix = np.array([models_dict[t].feature_importances_ for t in TARGETS])
    avg_imp    = imp_matrix.mean(axis=0)
    std_imp    = imp_matrix.std(axis=0)
 
    # Sort descending
    order     = np.argsort(avg_imp)
    sorted_feats = [FEATURE_COLS[i] for i in order]
    sorted_imp   = avg_imp[order]
    sorted_std   = std_imp[order]
 
    ax.barh(sorted_feats, sorted_imp, xerr=sorted_std,
            color=color, alpha=0.85, edgecolor='white',
            error_kw={'linewidth': 1, 'color': 'gray', 'capsize': 3})
    ax.set_xlabel('Mean importance (avg across 3 targets)')
    ax.set_title(f'{name}\n{"🏆 winner" if name == winner_name else "baseline"}',
                 fontweight='bold')
    ax.set_xlim(0, sorted_imp.max() * 1.25)
    for i, (val, std) in enumerate(zip(sorted_imp, sorted_std)):
        ax.text(val + std + 0.002, i, f'{val:.3f}', va='center', fontsize=8)
 
plt.tight_layout()
plt.savefig('assets/figures/feature_importances.png', dpi=150, bbox_inches='tight')
print("Saved: assets/figures/feature_importances.png")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 6 – MAPE by download speed tier for ALL competing models
# Lets you see whether RF beats GB/NN consistently across tiers
# ─────────────────────────────────────────────────────────────────────────────
bins        = [0, 50, 150, 300, np.inf]
tier_labels = ['Low\n(<50 Mbps)', 'Medium\n(50–150)', 'High\n(150–300)', 'Very High\n(>300)']
dl_true     = y_test[:, 0]
tiers       = pd.cut(dl_true, bins=bins, labels=tier_labels)
 
tier_mape_all = {name: [] for name in competing_names}
tier_ns       = []
for tier in tier_labels:
    mask = tiers == tier
    tier_ns.append(int(mask.sum()))
    for name, preds in [('Gradient Boosting', gb_preds),
                         ('Random Forest',     rf_preds),
                         ('Neural Network (ours)', nn_preds)]:
        val = mape(dl_true[mask], preds[mask, 0]) if mask.sum() > 0 else 0.0
        tier_mape_all[name].append(val)
 
x_tier = np.arange(len(tier_labels))
width_t = 0.26
 
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('MAPE by Download Speed Tier — all models compared\n'
             '(shows where each model is proportionally most/least accurate)',
             fontsize=11)
 
for j, name in enumerate(competing_names):
    offset = (j - len(competing_names)/2 + 0.5) * width_t
    bars   = ax.bar(x_tier + offset, tier_mape_all[name], width_t * 0.9,
                    label=f"{'🏆 ' if name == winner_name else ''}{name}",
                    color=BAR_COLORS[name], alpha=0.88, edgecolor='white')
    for bar, val in zip(bars, tier_mape_all[name]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f'{val:.0f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
 
# Annotate sample counts on x-axis
x_tick_labels = [f"{t.replace(chr(10),' ')}\n(n={n})" for t, n in zip(tier_labels, tier_ns)]
ax.set_xticks(x_tier)
ax.set_xticklabels(x_tick_labels, fontsize=9)
ax.set_ylabel('MAPE (%)  ← lower is better')
ax.axhline(30, color='green',  linewidth=1, linestyle='--', alpha=0.5, label='30% (good)')
ax.axhline(60, color='orange', linewidth=1, linestyle='--', alpha=0.5, label='60% (moderate)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('assets/figures/mape_by_tier_all_models.png', dpi=150)
print("Saved: assets/figures/mape_by_tier_all_models.png")
 
# ─────────────────────────────────────────────────────────────────────────────
# PLOT 7 – Residuals for the winner model
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'🏆 {winner_name} — Residual Distributions\n'
             '(centred at 0 = no systematic bias)', fontsize=12)
 
for i, (target, ax) in enumerate(zip(TARGETS, axes)):
    residuals = y_test[:, i] - winner_preds[:, i]
    ax.hist(residuals, bins=50, color=BAR_COLORS[winner_name],
            edgecolor='white', alpha=0.82)
    ax.axvline(0, color='red',    linewidth=1.5, linestyle='--', label='Zero (no bias)')
    ax.axvline(residuals.mean(), color='orange', linewidth=1.5, linestyle='--',
               label=f'Mean = {residuals.mean():.1f}')
    ax.set_xlabel(f'Residual ({TARGET_UNITS[target]})')
    ax.set_ylabel('Count')
    ax.set_title(target)
    ax.legend(fontsize=8)
 
plt.tight_layout()
plt.savefig('assets/figures/winner_residuals.png', dpi=150)
print("Saved: assets/figures/winner_residuals.png")
 
# ─────────────────────────────────────────────────────────────────────────────
# Console: copy-paste summary for README / presentation
# ─────────────────────────────────────────────────────────────────────────────
print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESULTS SUMMARY  (copy into README or presentation slides)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🏆  WINNER: {winner_name}
 
  Download speed (avg_d_mbps):
    {winner_name:<26} RMSE={competing[winner_name]['avg_d_mbps']['RMSE']} Mbps  MAE={competing[winner_name]['avg_d_mbps']['MAE']} Mbps  MAPE={competing[winner_name]['avg_d_mbps']['MAPE']}%  R²={competing[winner_name]['avg_d_mbps']['R2']}
    Gradient Boosting          RMSE={gb_results['avg_d_mbps']['RMSE']} Mbps  MAE={gb_results['avg_d_mbps']['MAE']} Mbps  MAPE={gb_results['avg_d_mbps']['MAPE']}%  R²={gb_results['avg_d_mbps']['R2']}
    Neural Network             RMSE={nn_results['avg_d_mbps']['RMSE']} Mbps  MAE={nn_results['avg_d_mbps']['MAE']} Mbps  MAPE={nn_results['avg_d_mbps']['MAPE']}%  R²={nn_results['avg_d_mbps']['R2']}
    Dummy (floor)              RMSE={dummy_results['avg_d_mbps']['RMSE']} Mbps  MAE={dummy_results['avg_d_mbps']['MAE']} Mbps  MAPE={dummy_results['avg_d_mbps']['MAPE']}%  R²={dummy_results['avg_d_mbps']['R2']}
 
  Upload speed (avg_u_mbps):
    {winner_name:<26} RMSE={competing[winner_name]['avg_u_mbps']['RMSE']} Mbps  MAE={competing[winner_name]['avg_u_mbps']['MAE']} Mbps  MAPE={competing[winner_name]['avg_u_mbps']['MAPE']}%  R²={competing[winner_name]['avg_u_mbps']['R2']}
    Gradient Boosting          RMSE={gb_results['avg_u_mbps']['RMSE']} Mbps  MAE={gb_results['avg_u_mbps']['MAE']} Mbps  MAPE={gb_results['avg_u_mbps']['MAPE']}%  R²={gb_results['avg_u_mbps']['R2']}
    Neural Network             RMSE={nn_results['avg_u_mbps']['RMSE']} Mbps  MAE={nn_results['avg_u_mbps']['MAE']} Mbps  MAPE={nn_results['avg_u_mbps']['MAPE']}%  R²={nn_results['avg_u_mbps']['R2']}
 
  Latency (avg_lat_ms):
    {winner_name:<26} RMSE={competing[winner_name]['avg_lat_ms']['RMSE']} ms  MAE={competing[winner_name]['avg_lat_ms']['MAE']} ms  MAPE={competing[winner_name]['avg_lat_ms']['MAPE']}%  R²={competing[winner_name]['avg_lat_ms']['R2']}
    Gradient Boosting          RMSE={gb_results['avg_lat_ms']['RMSE']} ms  MAE={gb_results['avg_lat_ms']['MAE']} ms  MAPE={gb_results['avg_lat_ms']['MAPE']}%  R²={gb_results['avg_lat_ms']['R2']}
    Neural Network             RMSE={nn_results['avg_lat_ms']['RMSE']} ms  MAE={nn_results['avg_lat_ms']['MAE']} ms  MAPE={nn_results['avg_lat_ms']['MAPE']}%  R²={nn_results['avg_lat_ms']['R2']}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")
 
print("✅  Evaluation complete.")
print("    Figures saved to: assets/figures/")
print(f"    Winner model    : {winner_name}")
print("    Winner artifacts: outputs/models/best_model_name.txt")
if winner_name in ('Random Forest', 'Gradient Boosting'):
    print("                      outputs/models/best_model_sklearn.pkl")
 

Train: 4397 rows  |  Val: 776 rows  |  Test: 913 rows

Fitting Dummy predictor …
Fitting Gradient Boosting (200 trees × 3 targets) …
  GB [avg_d_mbps] done
  GB [avg_u_mbps] done
  GB [avg_lat_ms] done
Fitting Random Forest (300 trees × 3 targets) …
  RF [avg_d_mbps] done
  RF [avg_u_mbps] done
  RF [avg_lat_ms] done
Loading Neural Network weights …

Computing metrics …

═════════════════════════════════════════════════════════════════════════════════════
Target             Model                            RMSE      MAE     MAPE       R²
═════════════════════════════════════════════════════════════════════════════════════
avg_d_mbps         Dummy (mean)                  121.801   93.775   158.1%  -0.0081
avg_d_mbps         Gradient Boosting              83.436   50.785    66.6%   0.5269
avg_d_mbps         Random Forest                  82.044   50.294    71.8%   0.5426
avg_d_mbps         Neural Network (ours)          84.699   52.696    79.3%   0.5125
──────────────────────────────────

C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\3543334138.py:443: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.savefig('assets/figures/model_comparison_all_metrics.png', dpi=150, bbox_inches='tight')


Saved: assets/figures/model_comparison_all_metrics.png


C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\3543334138.py:488: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\3543334138.py:489: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.savefig('assets/figures/rank_scoreboard.png', dpi=150, bbox_inches='tight')


Saved: assets/figures/rank_scoreboard.png


...

Saved: assets/figures/feature_importances.png


C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\3543334138.py:606: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\3543334138.py:607: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.savefig('assets/figures/mape_by_tier_all_models.png', dpi=150)


Saved: assets/figures/mape_by_tier_all_models.png


C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\3543334138.py:629: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\3543334138.py:630: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.savefig('assets/figures/winner_residuals.png', dpi=150)


Saved: assets/figures/winner_residuals.png

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESULTS SUMMARY  (copy into README or presentation slides)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🏆  WINNER: Random Forest

  Download speed (avg_d_mbps):
    Random Forest              RMSE=82.044 Mbps  MAE=50.294 Mbps  MAPE=71.8%  R²=0.5426
    Gradient Boosting          RMSE=83.436 Mbps  MAE=50.785 Mbps  MAPE=66.6%  R²=0.5269
    Neural Network             RMSE=84.699 Mbps  MAE=52.696 Mbps  MAPE=79.3%  R²=0.5125
    Dummy (floor)              RMSE=121.801 Mbps  MAE=93.775 Mbps  MAPE=158.1%  R²=-0.0081

  Upload speed (avg_u_mbps):
    Random Forest              RMSE=9.463 Mbps  MAE=6.378 Mbps  MAPE=67.4%  R²=0.4491
    Gradient Boosting          RMSE=9.573 Mbps  MAE=6.45 Mbps  MAPE=63.5%  R²=0.4362
    Neural Network             RMSE=9.843 Mbps  MAE=6.703 Mbps  MAPE=73.6%  R²=0.404

  Latency (avg_lat_ms):
    Random Forest   

In [73]:
# 04_evaluation.py
"""
04_evaluation.py  –  Model Evaluation & Baseline Comparison
=============================================================
Run this after train.py has completed and saved artifacts.

Models evaluated:
  1. Dummy (mean predictor)
  2. Gradient Boosting
  3. Random Forest
  4. Neural Network (ours)
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pickle
import json
import sys
import os
import joblib

sys.path.insert(0, os.path.dirname(os.path.abspath('scripts/architecture.py')))
from architecture import NeuralNetworkRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.dummy import DummyRegressor

os.makedirs('assets/figures', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)

TARGETS = ['avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms']
TARGET_UNITS = {'avg_d_mbps': 'Mbps', 'avg_u_mbps': 'Mbps', 'avg_lat_ms': 'ms'}
RANDOM_SEED = 42


# ─────────────────────────────────────────────────────────────────────────────
# Metric helpers
# ─────────────────────────────────────────────────────────────────────────────
def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.abs(y_true) > eps
    if valid.sum() == 0:
        return float('nan')
    return float(np.mean(np.abs(y_true[valid] - y_pred[valid]) / np.abs(y_true[valid])) * 100)


def all_metrics(y_true, y_pred):
    return {
        'RMSE': round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 3),
        'MAE':  round(float(mean_absolute_error(y_true, y_pred)), 3),
        'MAPE': round(mape(y_true, y_pred), 1),
        'R2':   round(float(r2_score(y_true, y_pred)), 4),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Load artifacts
# ─────────────────────────────────────────────────────────────────────────────
with open('outputs/models/scaler_X.pkl', 'rb') as f:
    scaler_X = pickle.load(f)

with open('outputs/models/scalers_y.pkl', 'rb') as f:
    scalers_y = pickle.load(f)

with open('outputs/models/feature_cols.json', 'r') as f:
    FEATURE_COLS = json.load(f)


# ─────────────────────────────────────────────────────────────────────────────
# Recreate exact train/val/test splits
# ─────────────────────────────────────────────────────────────────────────────
features = pd.read_csv('data/processed/features_engineered.csv')
features = features.dropna(subset=FEATURE_COLS + TARGETS).reset_index(drop=True)

X = features[FEATURE_COLS].values.astype(float)
y = features[TARGETS].values.astype(float)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_SEED, shuffle=True
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, random_state=RANDOM_SEED, shuffle=True
)

X_train_s = scaler_X.transform(X_train)
X_test_s = scaler_X.transform(X_test)

print(f"Train: {X_train.shape[0]} rows  |  Val: {X_val.shape[0]} rows  |  Test: {X_test.shape[0]} rows")


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 1 – Dummy
# ─────────────────────────────────────────────────────────────────────────────
print("\nFitting Dummy predictor ...")
dummy_preds = np.zeros_like(y_test)

for i, target in enumerate(TARGETS):
    d = DummyRegressor(strategy='mean')
    d.fit(X_train_s, y_train[:, i])
    dummy_preds[:, i] = d.predict(X_test_s)


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 2 – Gradient Boosting
# ─────────────────────────────────────────────────────────────────────────────
print("Fitting Gradient Boosting (200 trees × 3 targets) ...")
gb_preds = np.zeros_like(y_test)
gb_models = {}

for i, target in enumerate(TARGETS):
    gb = GradientBoostingRegressor(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=4,
        random_state=RANDOM_SEED,
    )
    gb.fit(X_train_s, y_train[:, i])
    gb_models[target] = gb
    gb_preds[:, i] = gb.predict(X_test_s)
    print(f"  GB [{target}] done")


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 3 – Random Forest
# ─────────────────────────────────────────────────────────────────────────────
print("Fitting Random Forest (300 trees × 3 targets) ...")
rf_preds = np.zeros_like(y_test)
rf_models = {}

for i, target in enumerate(TARGETS):
    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=4,
        max_features='sqrt',
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    rf.fit(X_train_s, y_train[:, i])
    rf_models[target] = rf
    rf_preds[:, i] = rf.predict(X_test_s)
    print(f"  RF [{target}] done")


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 4 – Neural Network
# ─────────────────────────────────────────────────────────────────────────────
print("Loading Neural Network weights ...")
weights = np.load('outputs/models/nn_weights.npz')
nn_model = NeuralNetworkRegressor(input_dim=X_test_s.shape[1])
nn_model.load_weights(weights)

y_pred_s = nn_model.predict(X_test_s)
nn_preds = np.zeros_like(y_test)

for i, target in enumerate(TARGETS):
    nn_preds[:, i] = (
        scalers_y[target]
        .inverse_transform(y_pred_s[:, i].reshape(-1, 1))
        .ravel()
    )


# ─────────────────────────────────────────────────────────────────────────────
# Compute metrics
# ─────────────────────────────────────────────────────────────────────────────
print("\nComputing metrics ...")

all_results = {
    'Dummy (mean)':          {t: all_metrics(y_test[:, i], dummy_preds[:, i]) for i, t in enumerate(TARGETS)},
    'Gradient Boosting':     {t: all_metrics(y_test[:, i], gb_preds[:, i])    for i, t in enumerate(TARGETS)},
    'Random Forest':         {t: all_metrics(y_test[:, i], rf_preds[:, i])    for i, t in enumerate(TARGETS)},
    'Neural Network (ours)': {t: all_metrics(y_test[:, i], nn_preds[:, i])    for i, t in enumerate(TARGETS)},
}

dummy_results = all_results['Dummy (mean)']
gb_results = all_results['Gradient Boosting']
rf_results = all_results['Random Forest']
nn_results = all_results['Neural Network (ours)']


# ─────────────────────────────────────────────────────────────────────────────
# Print comparison table
# ─────────────────────────────────────────────────────────────────────────────
W = 85
print(f"\n{'═' * W}")
print(f"{'Target':<18} {'Model':<28} {'RMSE':>8} {'MAE':>8} {'MAPE':>8} {'R²':>8}")
print(f"{'═' * W}")

model_order = list(all_results.keys())
for target in TARGETS:
    for name in model_order:
        r = all_results[name][target]
        print(
            f"{target:<18} {name:<28} "
            f"{r['RMSE']:>8.3f} {r['MAE']:>8.3f} "
            f"{r['MAPE']:>7.1f}% {r['R2']:>8.4f}"
        )
    print(f"{'─' * W}")


# ─────────────────────────────────────────────────────────────────────────────
# Automatic winner selection
# ─────────────────────────────────────────────────────────────────────────────
competing = {k: v for k, v in all_results.items() if k != 'Dummy (mean)'}
model_names = list(competing.keys())

METRIC_WEIGHTS = [
    ('MAPE', 3, True),
    ('R2',   2, False),
    ('RMSE', 1, True),
    ('MAE',  1, True),
]

total_rank = {name: 0 for name in model_names}

for target in TARGETS:
    for metric, weight, lower_better in METRIC_WEIGHTS:
        ordered = sorted(
            model_names,
            key=lambda n: competing[n][target][metric],
            reverse=not lower_better
        )
        for pos, name in enumerate(ordered, start=1):
            total_rank[name] += pos * weight

ranked_models = sorted(total_rank.items(), key=lambda x: x[1])
winner_name, winner_score = ranked_models[0]

print(f"\n{'━' * W}")
print("  🏆  AUTOMATIC WINNER SELECTION")
print(f"{'━' * W}")

for rank_pos, (name, score) in enumerate(ranked_models, start=1):
    medal = ['🥇', '🥈', '🥉'][rank_pos - 1] if rank_pos <= 3 else '  '
    arrow = '  ← SELECTED FOR DEPLOYMENT' if rank_pos == 1 else ''
    d_mape = competing[name]['avg_d_mbps']['MAPE']
    u_mape = competing[name]['avg_u_mbps']['MAPE']
    l_mape = competing[name]['avg_lat_ms']['MAPE']
    print(
        f"  {medal} #{rank_pos}  {name:<28} "
        f"weighted score={score:>3}  "
        f"MAPE: dl={d_mape:.1f}% up={u_mape:.1f}% lat={l_mape:.1f}%{arrow}"
    )

print(f"\n{'─' * W}")
print(f"  WINNER: {winner_name}")
print(f"{'─' * W}")


# ─────────────────────────────────────────────────────────────────────────────
# SAVE ALL MODELS + WINNER NAME + METRICS
# ─────────────────────────────────────────────────────────────────────────────
joblib.dump(gb_models, 'outputs/models/gb_models.pkl')
print("  Gradient Boosting saved → outputs/models/gb_models.pkl")

joblib.dump(rf_models, 'outputs/models/rf_models.pkl')
print("  Random Forest saved    → outputs/models/rf_models.pkl")

with open('outputs/models/best_model_name.txt', 'w') as f:
    f.write(winner_name)
print(f"  Winner saved           → outputs/models/best_model_name.txt ({winner_name})")

eval_summary = {
    name: {target: all_results[name][target] for target in TARGETS}
    for name in model_order
}
with open('outputs/models/eval_all_models.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)
print("  All metrics saved      → outputs/models/eval_all_models.json")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1 – Loss curves
# ─────────────────────────────────────────────────────────────────────────────
train_losses = np.load('outputs/models/train_losses.npy')
val_losses = np.load('outputs/models/val_losses.npy')

plt.figure(figsize=(9, 4))
plt.plot(train_losses, label='Train MSE (normalised)', linewidth=1.5, color='steelblue')
plt.plot(val_losses, label='Val MSE (normalised)', linewidth=1.5, color='tomato', alpha=0.85)
plt.xlabel('Epoch')
plt.ylabel('MSE (normalised scale)')
plt.title('Neural Network — Training vs Validation Loss Curves')
plt.legend()
plt.tight_layout()
plt.savefig('assets/figures/loss_curves.png', dpi=150)
print("\nSaved: assets/figures/loss_curves.png")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2 – Metric comparison
# ─────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 11))
gs = gridspec.GridSpec(2, 2, hspace=0.50, wspace=0.38)
ax_list = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]
fig.suptitle(
    'All Models — Full Metric Comparison\n'
    '(lower RMSE / MAE / MAPE = better  ·  higher R² = better  ·  🏆 = winner)',
    fontsize=13,
    fontweight='bold'
)

metric_specs = [
    ('RMSE', 'RMSE (original units)', True, 'lower is better'),
    ('MAE',  'MAE (original units)', True, 'lower is better'),
    ('MAPE', 'MAPE (%)', True, 'lower % = better'),
    ('R2',   'R² (0–1)', False, 'higher is better'),
]

BAR_COLORS = {
    'Dummy (mean)': '#aaaaaa',
    'Gradient Boosting': '#f0ad4e',
    'Random Forest': '#2a9d8f',
    'Neural Network (ours)': '#2196F3',
}

x = np.arange(len(TARGETS))
n_models = len(all_results)
total_w = 0.8
width = total_w / n_models

for ax, (metric, ylabel, lower_better, note) in zip(ax_list, metric_specs):
    for j, name in enumerate(model_order):
        vals = [all_results[name][t][metric] for t in TARGETS]
        offset = (j - n_models / 2 + 0.5) * width
        label = f"{'🏆 ' if name == winner_name else ''}{name}"

        bars = ax.bar(
            x + offset, vals, width * 0.92,
            label=label,
            color=BAR_COLORS[name],
            alpha=0.90 if name != 'Dummy (mean)' else 0.45,
            edgecolor='white',
            linewidth=0.5
        )

        for bar, val in zip(bars, vals):
            suffix = '%' if metric == 'MAPE' else ''
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + (0.005 if metric == 'R2' else abs(bar.get_height()) * 0.04 + 0.15),
                f'{val:.1f}{suffix}',
                ha='center',
                va='bottom',
                fontsize=6.5,
                fontweight='bold'
            )

    ax.set_xticks(x)
    ax.set_xticklabels(TARGETS, rotation=12, ha='right', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{metric} — {note}', fontsize=9.5)
    ax.legend(fontsize=7.5, loc='upper right')

plt.savefig('assets/figures/model_comparison_all_metrics.png', dpi=150, bbox_inches='tight')
print("Saved: assets/figures/model_comparison_all_metrics.png")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT 3 – Winner actual vs predicted
# ─────────────────────────────────────────────────────────────────────────────
winner_preds_map = {
    'Gradient Boosting': gb_preds,
    'Random Forest': rf_preds,
    'Neural Network (ours)': nn_preds,
}
winner_preds = winner_preds_map[winner_name]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'🏆 Winner: {winner_name} — Predicted vs Actual (Test Set)', fontsize=13)

for i, (target, ax) in enumerate(zip(TARGETS, axes)):
    r = competing[winner_name][target]
    unit = TARGET_UNITS[target]
    color = BAR_COLORS[winner_name]

    ax.scatter(y_test[:, i], winner_preds[:, i], alpha=0.30, s=8, color=color)
    lims = [
        min(y_test[:, i].min(), winner_preds[:, i].min()) * 0.95,
        max(y_test[:, i].max(), winner_preds[:, i].max()) * 1.05
    ]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
    ax.set_xlabel(f'Actual ({unit})')
    ax.set_ylabel(f'Predicted ({unit})')
    ax.set_title(f'{target}\nR²={r["R2"]}  |  MAPE={r["MAPE"]:.1f}%')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('assets/figures/winner_actual_vs_predicted.png', dpi=150)
print("Saved: assets/figures/winner_actual_vs_predicted.png")


print("\n✅ Evaluation complete.")
print("   Figures saved to: assets/figures/")
print(f"   Winner model    : {winner_name}")
print("   Saved models    : gb_models.pkl, rf_models.pkl, nn_weights.npz")

Train: 4397 rows  |  Val: 776 rows  |  Test: 913 rows

Fitting Dummy predictor ...
Fitting Gradient Boosting (200 trees × 3 targets) ...
  GB [avg_d_mbps] done
  GB [avg_u_mbps] done
  GB [avg_lat_ms] done
Fitting Random Forest (300 trees × 3 targets) ...
  RF [avg_d_mbps] done
  RF [avg_u_mbps] done
  RF [avg_lat_ms] done
Loading Neural Network weights ...

Computing metrics ...

═════════════════════════════════════════════════════════════════════════════════════
Target             Model                            RMSE      MAE     MAPE       R²
═════════════════════════════════════════════════════════════════════════════════════
avg_d_mbps         Dummy (mean)                  121.801   93.775   158.1%  -0.0081
avg_d_mbps         Gradient Boosting              84.866   51.279    65.8%   0.5106
avg_d_mbps         Random Forest                  82.044   50.294    71.8%   0.5426
avg_d_mbps         Neural Network (ours)          84.699   52.696    79.3%   0.5125
────────────────────────

C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\3472020926.py:362: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.savefig('assets/figures/model_comparison_all_metrics.png', dpi=150, bbox_inches='tight')


Saved: assets/figures/model_comparison_all_metrics.png


C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\3472020926.py:395: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\3472020926.py:396: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.savefig('assets/figures/winner_actual_vs_predicted.png', dpi=150)


Saved: assets/figures/winner_actual_vs_predicted.png

✅ Evaluation complete.
   Figures saved to: assets/figures/
   Winner model    : Random Forest
   Saved models    : gb_models.pkl, rf_models.pkl, nn_weights.npz


In [74]:
# ─────────────────────────────────────────────────────────────────────────────
# PLOT – MAPE comparison (4 models, like attached figure)
# ─────────────────────────────────────────────────────────────────────────────
plt.figure(figsize=(15, 7))

x = np.arange(len(TARGETS))
width = 0.2

model_order = [
    'Dummy (mean)',
    'Gradient Boosting',
    'Random Forest',
    'Neural Network (ours)'
]

display_names = {
    'Dummy (mean)': 'Dummy',
    'Gradient Boosting': 'Gradient Boosting',
    'Random Forest': 'Random Forest',
    'Neural Network (ours)': 'Neural Network'
}

BAR_COLORS = {
    'Dummy (mean)': '#d86a63',
    'Gradient Boosting': '#e8b25f',
    'Random Forest': '#5aa7a2',
    'Neural Network (ours)': '#74b96f',
}

for i, model_name in enumerate(model_order):
    values = [all_results[model_name][target]['MAPE'] for target in TARGETS]

    bars = plt.bar(
        x + (i - 1.5) * width,
        values,
        width=width,
        label=display_names[model_name],
        color=BAR_COLORS[model_name]
    )

    # value labels above bars
    for bar, val in zip(bars, values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{val:.1f}%",
            ha='center',
            va='bottom',
            fontsize=11
        )

plt.xticks(x, TARGETS, rotation=12)
plt.ylabel("MAPE (%)", fontsize=17)
plt.title(
    "Model Comparison: Mean Absolute Percentage Error (MAPE)\n"
    "Lower % = more accurate relative to the true value",
    fontsize=22,
    pad=10
)
plt.legend(fontsize=17, loc='upper right')
plt.grid(axis='y', linestyle='--', alpha=0.25)
plt.tight_layout()

plt.savefig('assets/figures/model_comparison_mape_4_models.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: assets/figures/model_comparison_mape_4_models.png")

Saved: assets/figures/model_comparison_mape_4_models.png


C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\3542639483.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [72]:
# 04_evaluation.py
"""
04_evaluation.py  –  Model Evaluation & Baseline Comparison
=============================================================
Run this after train.py has completed and saved artifacts.

Models evaluated:
  1. Dummy (mean predictor)
  2. Gradient Boosting
  3. Random Forest
  4. Neural Network (ours)
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pickle
import json
import sys
import os
import joblib

sys.path.insert(0, os.path.dirname(os.path.abspath('scripts/architecture.py')))
from architecture import NeuralNetworkRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.dummy import DummyRegressor

os.makedirs('assets/figures', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)

TARGETS = ['avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms']
TARGET_UNITS = {'avg_d_mbps': 'Mbps', 'avg_u_mbps': 'Mbps', 'avg_lat_ms': 'ms'}
RANDOM_SEED = 42


# ─────────────────────────────────────────────────────────────────────────────
# Metric helpers
# ─────────────────────────────────────────────────────────────────────────────
def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.abs(y_true) > eps
    if valid.sum() == 0:
        return float('nan')
    return float(np.mean(np.abs(y_true[valid] - y_pred[valid]) / np.abs(y_true[valid])) * 100)


def all_metrics(y_true, y_pred):
    return {
        'RMSE': round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 3),
        'MAE':  round(float(mean_absolute_error(y_true, y_pred)), 3),
        'MAPE': round(mape(y_true, y_pred), 1),
        'R2':   round(float(r2_score(y_true, y_pred)), 4),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Load artifacts
# ─────────────────────────────────────────────────────────────────────────────
with open('outputs/models/scaler_X.pkl', 'rb') as f:
    scaler_X = pickle.load(f)

with open('outputs/models/scalers_y.pkl', 'rb') as f:
    scalers_y = pickle.load(f)

with open('outputs/models/feature_cols.json', 'r') as f:
    FEATURE_COLS = json.load(f)


# ─────────────────────────────────────────────────────────────────────────────
# Recreate exact train/val/test splits
# ─────────────────────────────────────────────────────────────────────────────
features = pd.read_csv('data/processed/features_engineered.csv')
features = features.dropna(subset=FEATURE_COLS + TARGETS).reset_index(drop=True)

X = features[FEATURE_COLS].values.astype(float)
y = features[TARGETS].values.astype(float)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_SEED, shuffle=True
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, random_state=RANDOM_SEED, shuffle=True
)

X_train_s = scaler_X.transform(X_train)
X_test_s = scaler_X.transform(X_test)

print(f"Train: {X_train.shape[0]} rows  |  Val: {X_val.shape[0]} rows  |  Test: {X_test.shape[0]} rows")


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 1 – Dummy
# ─────────────────────────────────────────────────────────────────────────────
print("\nFitting Dummy predictor ...")
dummy_preds = np.zeros_like(y_test)

for i, target in enumerate(TARGETS):
    d = DummyRegressor(strategy='mean')
    d.fit(X_train_s, y_train[:, i])
    dummy_preds[:, i] = d.predict(X_test_s)


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 2 – Gradient Boosting
# ─────────────────────────────────────────────────────────────────────────────
print("Fitting Gradient Boosting (200 trees × 3 targets) ...")
gb_preds = np.zeros_like(y_test)
gb_models = {}

for i, target in enumerate(TARGETS):
    gb = GradientBoostingRegressor(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=4,
        random_state=RANDOM_SEED,
    )
    gb.fit(X_train_s, y_train[:, i])
    gb_models[target] = gb
    gb_preds[:, i] = gb.predict(X_test_s)
    print(f"  GB [{target}] done")


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 3 – Random Forest
# ─────────────────────────────────────────────────────────────────────────────
print("Fitting Random Forest (300 trees × 3 targets) ...")
rf_preds = np.zeros_like(y_test)
rf_models = {}

for i, target in enumerate(TARGETS):
    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=4,
        max_features='sqrt',
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    rf.fit(X_train_s, y_train[:, i])
    rf_models[target] = rf
    rf_preds[:, i] = rf.predict(X_test_s)
    print(f"  RF [{target}] done")


# ─────────────────────────────────────────────────────────────────────────────
# MODEL 4 – Neural Network
# ─────────────────────────────────────────────────────────────────────────────
print("Loading Neural Network weights ...")
weights = np.load('outputs/models/nn_weights.npz')
nn_model = NeuralNetworkRegressor(input_dim=X_test_s.shape[1])
nn_model.load_weights(weights)

y_pred_s = nn_model.predict(X_test_s)
nn_preds = np.zeros_like(y_test)

for i, target in enumerate(TARGETS):
    nn_preds[:, i] = (
        scalers_y[target]
        .inverse_transform(y_pred_s[:, i].reshape(-1, 1))
        .ravel()
    )


# ─────────────────────────────────────────────────────────────────────────────
# Compute metrics
# ─────────────────────────────────────────────────────────────────────────────
print("\nComputing metrics ...")

all_results = {
    'Dummy (mean)':          {t: all_metrics(y_test[:, i], dummy_preds[:, i]) for i, t in enumerate(TARGETS)},
    'Gradient Boosting':     {t: all_metrics(y_test[:, i], gb_preds[:, i])    for i, t in enumerate(TARGETS)},
    'Random Forest':         {t: all_metrics(y_test[:, i], rf_preds[:, i])    for i, t in enumerate(TARGETS)},
    'Neural Network (ours)': {t: all_metrics(y_test[:, i], nn_preds[:, i])    for i, t in enumerate(TARGETS)},
}

dummy_results = all_results['Dummy (mean)']
gb_results = all_results['Gradient Boosting']
rf_results = all_results['Random Forest']
nn_results = all_results['Neural Network (ours)']


# ─────────────────────────────────────────────────────────────────────────────
# Print comparison table
# ─────────────────────────────────────────────────────────────────────────────
W = 85
print(f"\n{'═' * W}")
print(f"{'Target':<18} {'Model':<28} {'RMSE':>8} {'MAE':>8} {'MAPE':>8} {'R²':>8}")
print(f"{'═' * W}")

model_order = list(all_results.keys())
for target in TARGETS:
    for name in model_order:
        r = all_results[name][target]
        print(
            f"{target:<18} {name:<28} "
            f"{r['RMSE']:>8.3f} {r['MAE']:>8.3f} "
            f"{r['MAPE']:>7.1f}% {r['R2']:>8.4f}"
        )
    print(f"{'─' * W}")


# ─────────────────────────────────────────────────────────────────────────────
# Automatic winner selection
# ─────────────────────────────────────────────────────────────────────────────
competing = {k: v for k, v in all_results.items() if k != 'Dummy (mean)'}
model_names = list(competing.keys())

METRIC_WEIGHTS = [
    ('MAPE', 3, True),
    ('R2',   2, False),
    ('RMSE', 1, True),
    ('MAE',  1, True),
]

total_rank = {name: 0 for name in model_names}

for target in TARGETS:
    for metric, weight, lower_better in METRIC_WEIGHTS:
        ordered = sorted(
            model_names,
            key=lambda n: competing[n][target][metric],
            reverse=not lower_better
        )
        for pos, name in enumerate(ordered, start=1):
            total_rank[name] += pos * weight

ranked_models = sorted(total_rank.items(), key=lambda x: x[1])
winner_name, winner_score = ranked_models[0]

print(f"\n{'━' * W}")
print("  🏆  AUTOMATIC WINNER SELECTION")
print(f"{'━' * W}")

for rank_pos, (name, score) in enumerate(ranked_models, start=1):
    medal = ['🥇', '🥈', '🥉'][rank_pos - 1] if rank_pos <= 3 else '  '
    arrow = '  ← SELECTED FOR DEPLOYMENT' if rank_pos == 1 else ''
    d_mape = competing[name]['avg_d_mbps']['MAPE']
    u_mape = competing[name]['avg_u_mbps']['MAPE']
    l_mape = competing[name]['avg_lat_ms']['MAPE']
    print(
        f"  {medal} #{rank_pos}  {name:<28} "
        f"weighted score={score:>3}  "
        f"MAPE: dl={d_mape:.1f}% up={u_mape:.1f}% lat={l_mape:.1f}%{arrow}"
    )

print(f"\n{'─' * W}")
print(f"  WINNER: {winner_name}")
print(f"{'─' * W}")


# ─────────────────────────────────────────────────────────────────────────────
# SAVE ALL MODELS + WINNER NAME + METRICS
# ─────────────────────────────────────────────────────────────────────────────
joblib.dump(gb_models, 'outputs/models/gb_models.pkl')
print("  Gradient Boosting saved → outputs/models/gb_models.pkl")

joblib.dump(rf_models, 'outputs/models/rf_models.pkl')
print("  Random Forest saved    → outputs/models/rf_models.pkl")

with open('outputs/models/best_model_name.txt', 'w') as f:
    f.write(winner_name)
print(f"  Winner saved           → outputs/models/best_model_name.txt ({winner_name})")

eval_summary = {
    name: {target: all_results[name][target] for target in TARGETS}
    for name in model_order
}
with open('outputs/models/eval_all_models.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)
print("  All metrics saved      → outputs/models/eval_all_models.json")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT 1 – Loss curves
# ─────────────────────────────────────────────────────────────────────────────
train_losses = np.load('outputs/models/train_losses.npy')
val_losses = np.load('outputs/models/val_losses.npy')

plt.figure(figsize=(9, 4))
plt.plot(train_losses, label='Train MSE (normalised)', linewidth=1.5, color='steelblue')
plt.plot(val_losses, label='Val MSE (normalised)', linewidth=1.5, color='tomato', alpha=0.85)
plt.xlabel('Epoch')
plt.ylabel('MSE (normalised scale)')
plt.title('Neural Network — Training vs Validation Loss Curves')
plt.legend()
plt.tight_layout()
plt.savefig('assets/figures/loss_curves.png', dpi=150)
print("\nSaved: assets/figures/loss_curves.png")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2 – Metric comparison
# ─────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 11))
gs = gridspec.GridSpec(2, 2, hspace=0.50, wspace=0.38)
ax_list = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]
fig.suptitle(
    'All Models — Full Metric Comparison\n'
    '(lower RMSE / MAE / MAPE = better  ·  higher R² = better  ·  🏆 = winner)',
    fontsize=13,
    fontweight='bold'
)

metric_specs = [
    ('RMSE', 'RMSE (original units)', True, 'lower is better'),
    ('MAE',  'MAE (original units)', True, 'lower is better'),
    ('MAPE', 'MAPE (%)', True, 'lower % = better'),
    ('R2',   'R² (0–1)', False, 'higher is better'),
]

BAR_COLORS = {
    'Dummy (mean)': '#aaaaaa',
    'Gradient Boosting': '#f0ad4e',
    'Random Forest': '#2a9d8f',
    'Neural Network (ours)': '#2196F3',
}

x = np.arange(len(TARGETS))
n_models = len(all_results)
total_w = 0.8
width = total_w / n_models

for ax, (metric, ylabel, lower_better, note) in zip(ax_list, metric_specs):
    for j, name in enumerate(model_order):
        vals = [all_results[name][t][metric] for t in TARGETS]
        offset = (j - n_models / 2 + 0.5) * width
        label = f"{'🏆 ' if name == winner_name else ''}{name}"

        bars = ax.bar(
            x + offset, vals, width * 0.92,
            label=label,
            color=BAR_COLORS[name],
            alpha=0.90 if name != 'Dummy (mean)' else 0.45,
            edgecolor='white',
            linewidth=0.5
        )

        for bar, val in zip(bars, vals):
            suffix = '%' if metric == 'MAPE' else ''
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + (0.005 if metric == 'R2' else abs(bar.get_height()) * 0.04 + 0.15),
                f'{val:.1f}{suffix}',
                ha='center',
                va='bottom',
                fontsize=6.5,
                fontweight='bold'
            )

    ax.set_xticks(x)
    ax.set_xticklabels(TARGETS, rotation=12, ha='right', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{metric} — {note}', fontsize=9.5)
    ax.legend(fontsize=7.5, loc='upper right')

plt.savefig('assets/figures/model_comparison_all_metrics.png', dpi=150, bbox_inches='tight')
print("Saved: assets/figures/model_comparison_all_metrics.png")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT 3 – Winner actual vs predicted
# ─────────────────────────────────────────────────────────────────────────────
winner_preds_map = {
    'Gradient Boosting': gb_preds,
    'Random Forest': rf_preds,
    'Neural Network (ours)': nn_preds,
}
winner_preds = winner_preds_map[winner_name]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'🏆 Winner: {winner_name} — Predicted vs Actual (Test Set)', fontsize=13)

for i, (target, ax) in enumerate(zip(TARGETS, axes)):
    r = competing[winner_name][target]
    unit = TARGET_UNITS[target]
    color = BAR_COLORS[winner_name]

    ax.scatter(y_test[:, i], winner_preds[:, i], alpha=0.30, s=8, color=color)
    lims = [
        min(y_test[:, i].min(), winner_preds[:, i].min()) * 0.95,
        max(y_test[:, i].max(), winner_preds[:, i].max()) * 1.05
    ]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
    ax.set_xlabel(f'Actual ({unit})')
    ax.set_ylabel(f'Predicted ({unit})')
    ax.set_title(f'{target}\nR²={r["R2"]}  |  MAPE={r["MAPE"]:.1f}%')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('assets/figures/winner_actual_vs_predicted.png', dpi=150)
print("Saved: assets/figures/winner_actual_vs_predicted.png")


print("\n✅ Evaluation complete.")
print("   Figures saved to: assets/figures/")
print(f"   Winner model    : {winner_name}")
print("   Saved models    : gb_models.pkl, rf_models.pkl, nn_weights.npz")

Train: 4397 rows  |  Val: 776 rows  |  Test: 913 rows

Fitting Dummy predictor ...
Fitting Gradient Boosting (200 trees × 3 targets) ...
  GB [avg_d_mbps] done
  GB [avg_u_mbps] done
  GB [avg_lat_ms] done
Fitting Random Forest (300 trees × 3 targets) ...
  RF [avg_d_mbps] done
  RF [avg_u_mbps] done
  RF [avg_lat_ms] done
Loading Neural Network weights ...

Computing metrics ...

═════════════════════════════════════════════════════════════════════════════════════
Target             Model                            RMSE      MAE     MAPE       R²
═════════════════════════════════════════════════════════════════════════════════════
avg_d_mbps         Dummy (mean)                  121.801   93.775   158.1%  -0.0081
avg_d_mbps         Gradient Boosting              85.337   51.589    66.8%   0.5051
avg_d_mbps         Random Forest                  82.044   50.294    71.8%   0.5426
avg_d_mbps         Neural Network (ours)          84.699   52.696    79.3%   0.5125
────────────────────────

C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\2171363956.py:362: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.savefig('assets/figures/model_comparison_all_metrics.png', dpi=150, bbox_inches='tight')


Saved: assets/figures/model_comparison_all_metrics.png


C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\2171363956.py:395: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\fatma\AppData\Local\Temp\ipykernel_18816\2171363956.py:396: UserWarning: Glyph 127942 (\N{TROPHY}) missing from font(s) DejaVu Sans.
  plt.savefig('assets/figures/winner_actual_vs_predicted.png', dpi=150)


Saved: assets/figures/winner_actual_vs_predicted.png

✅ Evaluation complete.
   Figures saved to: assets/figures/
   Winner model    : Random Forest
   Saved models    : gb_models.pkl, rf_models.pkl, nn_weights.npz


In [71]:
from sklearn.metrics import mean_squared_error

for n in [200, 400, 600, 800, 1000]:
    model = GradientBoostingRegressor(
        n_estimators=n,
        learning_rate=0.05,
        max_depth=4,
        random_state=RANDOM_SEED
    )
    
    model.fit(X_train_s, y_train[:, i])
    preds = model.predict(X_val)

    print(n, mean_squared_error(y_val, preds))

ValueError: y_true and y_pred have different number of output (3!=1)

In [47]:
#predict

In [49]:
# scripts/predict.py
import numpy as np
import pandas as pd
import pickle, json
import sys, os
sys.path.insert(0, os.path.dirname('scripts/architecture.py'))
from architecture import NeuralNetworkRegressor

_MODEL     = None
_SCALER_X  = None
_SCALER_Y  = None
_FEAT_COLS = None

def _load_artifacts():
    global _MODEL, _SCALER_X, _SCALER_Y, _FEAT_COLS
    if _MODEL is not None:
        return
    base = os.path.join(os.path.dirname(__file__), '..', 'outputs', 'models')
    with open(f'{base}/scaler_X.pkl', 'rb') as f: _SCALER_X = pickle.load(f)
    with open(f'{base}/scaler_y.pkl', 'rb') as f: _SCALER_Y = pickle.load(f)
    with open(f'{base}/feature_cols.json')   as f: _FEAT_COLS = json.load(f)
    
    weights = np.load(f'{base}/nn_weights.npz')
    _MODEL = NeuralNetworkRegressor(input_dim=len(_FEAT_COLS), output_dim=3)
    for i, layer in enumerate(_MODEL.layers):
        layer.W = weights[f'layer_{i}_W']
        layer.b = weights[f'layer_{i}_b']

def predict_speeds(input_dict: dict) -> dict:
    """
    input_dict: must contain all keys in FEATURE_COLS
    returns: {'avg_d_mbps': float, 'avg_u_mbps': float, 'avg_lat_ms': float}
    """
    _load_artifacts()
    X = pd.DataFrame([input_dict])[_FEAT_COLS].values.astype(float)
    X_s = _SCALER_X.transform(X)
    y_s = _MODEL.predict(X_s)
    y   = _SCALER_Y.inverse_transform(y_s)[0]
    return {
        'avg_d_mbps': round(float(np.clip(y[0], 0, None)), 2),
        'avg_u_mbps': round(float(np.clip(y[1], 0, None)), 2),
        'avg_lat_ms': round(float(np.clip(y[2], 0, None)), 2),
    }

In [51]:
#Recommend for the towers

In [10]:
"""
recommend.py  -  Top Tower Placement Recommendations
=====================================================
Computes the top-N areas in Bahrain that most need a new tower, based on:
  - Performance weakness  (low download speed relative to max observed)
  - Demand score          (high test count = high user demand)
  - Tower scarcity        (few towers within 5 km)
  - Demand growth         (year-over-year test count growth)
 
All inputs come from the preprocessed features CSV so results stay
consistent with what the model was trained on.
 
Usage
-----
    from recommend import get_top_recommendations
    recs = get_top_recommendations(top_n=5)
    # returns list of dicts, one per recommended area
"""
 
import numpy as np
import pandas as pd
import os
 
# ── paths ─────────────────────────────────────────────────────────────────────
_HERE        = os.path.dirname(os.path.abspath('data/processed/features_engineered.csv'))
_FEATURES_CSV = os.path.join('data', 'processed', 'features_engineered.csv')
_TOWERS_CSV   = os.path.join(
    'data', 'bahrain_towers.csv')
 
# Weights for the priority score (must sum to 1.0)
_W_PERF     = 0.35   # performance weakness
_W_DEMAND   = 0.20   # demand (test count)
_W_SCARCITY = 0.35   # tower scarcity
_W_GROWTH   = 0.10   # demand growth trend
 
 
def _load_data():
    features = pd.read_csv(_FEATURES_CSV)
    towers   = pd.read_csv(_TOWERS_CSV)
    return features, towers
 
 
def _type_label(code: str) -> str:
    """Convert typeOfArea code to human-readable label."""
    mapping = {
        'PPL':  'Populated Place',
        'PPLX': 'Populated Locality',
        'HTL':  'Hotel / Tourism',
        'PT':   'Point',
        'ISL':  'Island',
        'ISLX': 'Island Locality',
        'ISLS': 'Islands',
        'PRT':  'Port',
        'TOWR': 'Tower / Industrial',
    }
    return mapping.get(str(code).strip(), code)
 
 
def get_top_recommendations(top_n: int = 5) -> list[dict]:
    """
    Returns a ranked list of the top_n areas recommended for new tower placement.
 
    Each entry contains:
      area, city, region, typeOfArea, typeOfArea_label,
      latitude, longitude,
      avg_d_mbps, avg_u_mbps, avg_lat_ms, tests,
      nearest_tower_distance_km, tower_count_5km,
      demand_growth_pct,
      perf_weakness_score, demand_score, scarcity_score, growth_score,
      priority_score  (0-100, higher = more urgent),
      rank, recommendation_summary
    """
    features, towers = _load_data()
 
    # ── 1. Aggregate to one row per area (latest year available) ──────────────
    latest_year = features['year'].max()
    # prefer latest year, fall back to all-time mean if area missing in latest
    latest = features[features['year'] == latest_year].copy()
 
    agg = {
        'latitude':                  'mean',
        'longitude':                 'mean',
        'avg_d_mbps':                'mean',
        'avg_u_mbps':                'mean',
        'avg_lat_ms':                'mean',
        'tests':                     'sum',
        'nearest_tower_distance_km': 'mean',
        'tower_count_1km':           'mean',
        'tower_count_2km':           'mean',
        'tower_count_5km':           'mean',
        'demand_growth_pct':         'mean',
        'typeOfArea':                'first',
        'city':                      'first',
        'region':                    'first',
    }
 
    area_df = latest.groupby('area').agg(agg).reset_index()
 
    # ── 2. Normalise each scoring dimension to [0, 1] ─────────────────────────
    def minmax(s):
        rng = s.max() - s.min()
        return (s - s.min()) / rng if rng > 0 else pd.Series(0.5, index=s.index)
 
    # Performance weakness: HIGH when download speed is LOW
    area_df['perf_weakness_score'] = 1.0 - minmax(area_df['avg_d_mbps'])
 
    # Demand: HIGH when test count is HIGH
    area_df['demand_score'] = minmax(area_df['tests'])
 
    # Scarcity: HIGH when few towers within 5 km  (invert tower_count_5km)
    area_df['scarcity_score'] = 1.0 - minmax(area_df['tower_count_5km'])
 
    # Growth: HIGH when demand_growth_pct is HIGH
    area_df['growth_score'] = minmax(area_df['demand_growth_pct'].clip(0, None))
 
    # ── 3. Weighted priority score (0-100) ────────────────────────────────────
    area_df['priority_score'] = (
        _W_PERF     * area_df['perf_weakness_score'] +
        _W_DEMAND   * area_df['demand_score']        +
        _W_SCARCITY * area_df['scarcity_score']      +
        _W_GROWTH   * area_df['growth_score']
    ) * 100
 
    # ── 4. Pick top N ─────────────────────────────────────────────────────────
    top = (area_df
           .nlargest(top_n, 'priority_score')
           .reset_index(drop=True))
 
    # ── 5. Build output records ───────────────────────────────────────────────
    results = []
    for rank, row in top.iterrows():
        type_label = _type_label(row['typeOfArea'])
 
        # Auto-generate a plain-English summary
        weakness = ''
        if row['perf_weakness_score'] > 0.7:
            weakness = 'significantly underperforming network speeds'
        elif row['perf_weakness_score'] > 0.4:
            weakness = 'below-average network speeds'
        else:
            weakness = 'moderate network speeds'
 
        demand_txt = ''
        if row['demand_score'] > 0.7:
            demand_txt = 'very high user demand'
        elif row['demand_score'] > 0.4:
            demand_txt = 'moderate-to-high user demand'
        else:
            demand_txt = 'growing user demand'
 
        summary = (
            f"{row['area']} ({type_label}) in {row['city']} shows {weakness} "
            f"with {demand_txt} ({int(row['tests']):,} tests recorded). "
            f"Average download is {row['avg_d_mbps']:.1f} Mbps with "
            f"{row['tower_count_5km']:.1f} towers within 5 km. "
            f"A new tower here would serve the most underserved users."
        )
 
        results.append({
            'rank':                      rank + 1,
            'area':                      row['area'],
            'city':                      row['city'],
            'region':                    row['region'],
            'typeOfArea':                row['typeOfArea'],
            'typeOfArea_label':          type_label,
            'latitude':                  round(row['latitude'], 6),
            'longitude':                 round(row['longitude'], 6),
            'avg_d_mbps':                round(row['avg_d_mbps'], 2),
            'avg_u_mbps':                round(row['avg_u_mbps'], 2),
            'avg_lat_ms':                round(row['avg_lat_ms'], 2),
            'tests':                     int(row['tests']),
            'nearest_tower_distance_km': round(row['nearest_tower_distance_km'], 3),
            'tower_count_1km':           round(row['tower_count_1km'], 1),
            'tower_count_2km':           round(row['tower_count_2km'], 1),
            'tower_count_5km':           round(row['tower_count_5km'], 1),
            'demand_growth_pct':         round(row['demand_growth_pct'] * 100, 2),
            'perf_weakness_score':       round(row['perf_weakness_score'], 4),
            'demand_score':              round(row['demand_score'], 4),
            'scarcity_score':            round(row['scarcity_score'], 4),
            'growth_score':              round(row['growth_score'], 4),
            'priority_score':            round(row['priority_score'], 2),
            'recommendation_summary':    summary,
        })
 
    return results
 
 
if __name__ == '__main__':
    recs = get_top_recommendations(top_n=5)
    for r in recs:
        print(f"\nTop {r['rank']}: {r['area']} — Priority Score: {r['priority_score']}")
        print(f"  {r['recommendation_summary']}")
 


Top 1: Ar Rifa` ash Sharqi — Priority Score: 77.49
  Ar Rifa` ash Sharqi (Populated Locality) in Ar Rifa` ash Sharqi shows below-average network speeds with very high user demand (1,912 tests recorded). Average download is 230.8 Mbps with 21.0 towers within 5 km. A new tower here would serve the most underserved users.

Top 2: Al Safir Hotel & Tower — Priority Score: 73.95
  Al Safir Hotel & Tower (Hotel / Tourism) in Az Zallaq shows below-average network speeds with moderate-to-high user demand (1,101 tests recorded). Average download is 326.0 Mbps with 15.9 towers within 5 km. A new tower here would serve the most underserved users.

Top 3: Jazirat `Ajirah — Priority Score: 69.21
  Jazirat `Ajirah (Island) in Jazirat `Ajirah shows significantly underperforming network speeds with growing user demand (5 tests recorded). Average download is 25.4 Mbps with 0.0 towers within 5 km. A new tower here would serve the most underserved users.

Top 4: Jusayrah — Priority Score: 68.42
  Jusayra

In [7]:
ecs = get_top_recommendations(top_n=5)

# convert list of dicts to dataframe
df_recs = pd.DataFrame(recs)

print(df_recs)

   rank                    area                 city                region  \
0     1     Ar Rifa` ash Sharqi  Ar Rifa` ash Sharqi  Southern Governorate   
1     2  Al Safir Hotel & Tower            Az Zallaq  Northern Governorate   
2     3         Jazirat `Ajirah      Jazirat `Ajirah  Southern Governorate   
3     4                Jusayrah             Jusayrah  Southern Governorate   
4     5           Hawar Islands        Jazirat Hawar  Southern Governorate   

  typeOfArea    typeOfArea_label   latitude  longitude  avg_d_mbps  \
0       PPLX  Populated Locality  26.119838  50.576062      230.80   
1        HTL     Hotel / Tourism  26.033984  50.514352      325.99   
2        ISL              Island  25.728158  50.798035       25.45   
3        PPL     Populated Place  25.880365  50.568848      127.14   
4       ISLS             Islands  25.641526  50.767822       56.09   

   avg_u_mbps  ...  tower_count_1km  tower_count_2km  tower_count_5km  \
0       28.73  ...              1.1  

In [8]:
recs = get_top_recommendations(top_n=5)
df_recs = pd.DataFrame(recs)

cols = [
    'rank', 'area', 'city', 'region', 'typeOfArea_label',
    'latitude', 'longitude',
    'avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms',
    'tests', 'nearest_tower_distance_km',
    'tower_count_1km', 'tower_count_2km', 'tower_count_5km',
    'demand_growth_pct', 'priority_score',
    'recommendation_summary'
]

print(df_recs[cols])

   rank                    area                 city                region  \
0     1     Ar Rifa` ash Sharqi  Ar Rifa` ash Sharqi  Southern Governorate   
1     2  Al Safir Hotel & Tower            Az Zallaq  Northern Governorate   
2     3         Jazirat `Ajirah      Jazirat `Ajirah  Southern Governorate   
3     4                Jusayrah             Jusayrah  Southern Governorate   
4     5           Hawar Islands        Jazirat Hawar  Southern Governorate   

     typeOfArea_label   latitude  longitude  avg_d_mbps  avg_u_mbps  \
0  Populated Locality  26.119838  50.576062      230.80       28.73   
1     Hotel / Tourism  26.033984  50.514352      325.99       43.34   
2              Island  25.728158  50.798035       25.45       10.98   
3     Populated Place  25.880365  50.568848      127.14       14.56   
4             Islands  25.641526  50.767822       56.09       21.92   

   avg_lat_ms  tests  nearest_tower_distance_km  tower_count_1km  \
0       18.38   1912                

In [12]:
recs = get_top_recommendations(top_n=5)
df = pd.DataFrame(recs)

# Select only needed columns
table_df = df[[
    "rank", "area", "region", "priority_score",
    "avg_d_mbps", "nearest_tower_distance_km",
    "tower_count_5km", "demand_growth_pct"
]]

# Rename columns (clean names)
table_df.columns = [
    "Rank", "Area", "Region", "Priority Score",
    "Avg Speed (Mbps)", "Nearest Tower (km)",
    "Towers 5km", "Demand Growth (%)"
]

# Round values
table_df["Priority Score"] = table_df["Priority Score"].round(1)
table_df["Avg Speed (Mbps)"] = table_df["Avg Speed (Mbps)"].round(1)
table_df["Nearest Tower (km)"] = table_df["Nearest Tower (km)"].round(2)
table_df["Demand Growth (%)"] = table_df["Demand Growth (%)"].round(1)

# Plot table
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

table = ax.table(
    cellText=table_df.values,
    colLabels=table_df.columns,
    loc='center',
    cellLoc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)

plt.title("Top 5 Tower Placement Recommendations", fontsize=14)
plt.tight_layout()

plt.savefig("assets/figures/top5_tower_priorities_table.png", dpi=150)
plt.show()

C:\Users\fatma\AppData\Local\Temp\ipykernel_27644\93894132.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_df["Priority Score"] = table_df["Priority Score"].round(1)
C:\Users\fatma\AppData\Local\Temp\ipykernel_27644\93894132.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_df["Avg Speed (Mbps)"] = table_df["Avg Speed (Mbps)"].round(1)
C:\Users\fatma\AppData\Local\Temp\ipykernel_27644\93894132.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFra

NameError: name 'plt' is not defined

In [13]:
table_df

,Rank,Area,Region,Priority Score,Avg Speed (Mbps),Nearest Tower (km),Towers 5km,Demand Growth (%)
0,1,Ar Rifa` ash Sharqi,Southern Governorate,77.5,230.8,1.25,21.0,-65.1
1,2,Al Safir Hotel & Tower,Northern Governorate,74.0,326.0,1.38,15.9,122.9
2,3,Jazirat `Ajirah,Southern Governorate,69.2,25.4,23.57,0.0,-77.3
3,4,Jusayrah,Southern Governorate,68.4,127.1,3.07,3.7,57.1
4,5,Hawar Islands,Southern Governorate,67.5,56.1,28.67,0.0,-96.0


In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ===== Put your save path here =====
SAVE_DIR = r"assets/figures"
os.makedirs(SAVE_DIR, exist_ok=True)

In [17]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [18]:
MODELS_DIR = 'outputs/models'
SAVE_DIR = r'assets/figures'
os.makedirs(SAVE_DIR, exist_ok=True)

In [19]:
feature_cols_path = os.path.join(MODELS_DIR, 'feature_cols2.json')

with open(feature_cols_path, 'r') as f:
    feature_cols = json.load(f)

targets = ['avg_d_mbps', 'avg_u_mbps', 'avg_lat_ms']

X_val = np.load(os.path.join(MODELS_DIR, 'X_val.npy'), allow_pickle=True)
y_val = np.load(os.path.join(MODELS_DIR, 'y_val.npy'), allow_pickle=True)

X_val = pd.DataFrame(X_val, columns=feature_cols)
y_val = pd.DataFrame(y_val, columns=targets)

print(X_val.shape, y_val.shape)

(776, 21) (776, 3)


In [20]:
lr_models = joblib.load(os.path.join(MODELS_DIR, 'lr_models.pkl'))
dt_models = joblib.load(os.path.join(MODELS_DIR, 'dt_models.pkl'))
rf_models = joblib.load(os.path.join(MODELS_DIR, 'rf_models.pkl'))
gb_models = joblib.load(os.path.join(MODELS_DIR, 'gb_models.pkl'))

In [21]:
model_dicts = {
    'Linear Regression': lr_models,
    'Decision Tree': dt_models,
    'Random Forest': rf_models,
    'Gradient Boosting': gb_models,
}

val_preds = {}

for model_name, models in model_dicts.items():
    pred_arr = np.zeros((len(X_val), len(targets)))
    
    for i, target in enumerate(targets):
        y_pred = models[target].predict(X_val)
        y_pred = np.clip(y_pred, 0, None)
        pred_arr[:, i] = y_pred
    
    val_preds[model_name] = pred_arr

print(val_preds.keys())

C:\Users\fatma\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
C:\Users\fatma\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
C:\Users\fatma\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


dict_keys(['Linear Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting'])


In [22]:
rows = []

for model_name, pred_arr in val_preds.items():
    for i, target in enumerate(targets):
        yt = y_val.iloc[:, i].values
        yp = pred_arr[:, i]
        valid = yt > 0
        
        rows.append({
            'Model': model_name,
            'Target': target,
            'MAPE': mean_absolute_percentage_error(yt[valid], yp[valid]) * 100,
            'R2': r2_score(yt, yp),
            'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'MAE': mean_absolute_error(yt, yp),
        })

metrics_df = pd.DataFrame(rows)
metrics_df = metrics_df.round(4)
metrics_df

,Model,Target,MAPE,R2,RMSE,MAE
0,Linear Regression,avg_d_mbps,91.1921,-1.3170,168.6882,133.6386
1,Linear Regression,avg_u_mbps,268.7872,-20.0924,63.4375,36.3171
2,Linear Regression,avg_lat_ms,146.2220,-1.4642,57.8910,34.1054
3,Decision Tree,avg_d_mbps,62.3210,0.5105,77.5329,49.2972
4,Decision Tree,avg_u_mbps,57.2387,0.3542,11.1005,6.8355
5,Decision Tree,avg_lat_ms,37.0666,0.1704,33.5896,10.7881
6,Random Forest,avg_d_mbps,57.8635,0.5702,72.6492,46.3784
7,Random Forest,avg_u_mbps,55.7732,0.4174,10.5428,6.3332
8,Random Forest,avg_lat_ms,34.8314,0.1913,33.1633,10.0220
9,Gradient Boosting,avg_d_mbps,94.6095,-1.7122,182.5080,145.3992


In [23]:
styled_metrics = (
    metrics_df.style
    .background_gradient(subset=['R2'], cmap='Greens')     # higher = better
    .background_gradient(subset=['RMSE', 'MAE', 'MAPE'], cmap='Reds_r')  # lower = better
    .format({
        'R2': '{:.4f}',
        'RMSE': '{:.2f}',
        'MAE': '{:.2f}',
        'MAPE': '{:.2f}'
    })
    .set_caption("Model Performance Comparison")
)

styled_metrics

,Model,Target,MAPE,R2,RMSE,MAE
0,Linear Regression,avg_d_mbps,91.19,-1.3170,168.69,133.64
1,Linear Regression,avg_u_mbps,268.79,-20.0924,63.44,36.32
2,Linear Regression,avg_lat_ms,146.22,-1.4642,57.89,34.11
3,Decision Tree,avg_d_mbps,62.32,0.5105,77.53,49.30
4,Decision Tree,avg_u_mbps,57.24,0.3542,11.10,6.84
5,Decision Tree,avg_lat_ms,37.07,0.1704,33.59,10.79
6,Random Forest,avg_d_mbps,57.86,0.5702,72.65,46.38
7,Random Forest,avg_u_mbps,55.77,0.4174,10.54,6.33
8,Random Forest,avg_lat_ms,34.83,0.1913,33.16,10.02
9,Gradient Boosting,avg_d_mbps,94.61,-1.7122,182.51,145.40


In [24]:
def highlight_best(s, metric):
    if metric in ['RMSE', 'MAE', 'MAPE']:
        return ['background-color: lightgreen' if v == s.min() else '' for v in s]
    else:  # R2
        return ['background-color: lightgreen' if v == s.max() else '' for v in s]

styled_metrics = metrics_df.style

for metric in ['RMSE', 'MAE', 'MAPE', 'R2']:
    styled_metrics = styled_metrics.apply(
        highlight_best,
        metric=metric,
        subset=[metric]
    )

styled_metrics = styled_metrics.format({
    'R2': '{:.4f}',
    'RMSE': '{:.2f}',
    'MAE': '{:.2f}',
    'MAPE': '{:.2f}'
})

styled_metrics

,Model,Target,MAPE,R2,RMSE,MAE
0,Linear Regression,avg_d_mbps,91.19,-1.3170,168.69,133.64
1,Linear Regression,avg_u_mbps,268.79,-20.0924,63.44,36.32
2,Linear Regression,avg_lat_ms,146.22,-1.4642,57.89,34.11
3,Decision Tree,avg_d_mbps,62.32,0.5105,77.53,49.30
4,Decision Tree,avg_u_mbps,57.24,0.3542,11.10,6.84
5,Decision Tree,avg_lat_ms,37.07,0.1704,33.59,10.79
6,Random Forest,avg_d_mbps,57.86,0.5702,72.65,46.38
7,Random Forest,avg_u_mbps,55.77,0.4174,10.54,6.33
8,Random Forest,avg_lat_ms,34.83,0.1913,33.16,10.02
9,Gradient Boosting,avg_d_mbps,94.61,-1.7122,182.51,145.40


In [25]:
def highlight_best(s, metric):
    if metric in ['RMSE', 'MAE', 'MAPE']:
        return ['background-color: #90EE90' if v == s.min() else '' for v in s]
    else:
        return ['background-color: #90EE90' if v == s.max() else '' for v in s]

styled_metrics = (
    metrics_df.style
    .background_gradient(subset=['R2'], cmap='Greens')
    .background_gradient(subset=['RMSE', 'MAE', 'MAPE'], cmap='Reds_r')
)

for metric in ['RMSE', 'MAE', 'MAPE', 'R2']:
    styled_metrics = styled_metrics.apply(
        highlight_best,
        metric=metric,
        subset=[metric]
    )

styled_metrics = styled_metrics.set_caption("Model Performance Comparison (Highlighted Best)")
styled_metrics

,Model,Target,MAPE,R2,RMSE,MAE
0,Linear Regression,avg_d_mbps,91.192100,-1.317000,168.688200,133.638600
1,Linear Regression,avg_u_mbps,268.787200,-20.092400,63.437500,36.317100
2,Linear Regression,avg_lat_ms,146.222000,-1.464200,57.891000,34.105400
3,Decision Tree,avg_d_mbps,62.321000,0.510500,77.532900,49.297200
4,Decision Tree,avg_u_mbps,57.238700,0.354200,11.100500,6.835500
5,Decision Tree,avg_lat_ms,37.066600,0.170400,33.589600,10.788100
6,Random Forest,avg_d_mbps,57.863500,0.570200,72.649200,46.378400
7,Random Forest,avg_u_mbps,55.773200,0.417400,10.542800,6.333200
8,Random Forest,avg_lat_ms,34.831400,0.191300,33.163300,10.022000
9,Gradient Boosting,avg_d_mbps,94.609500,-1.712200,182.508000,145.399200
